
# FlipItNews News Classification using NLP and DistilBERT
# By Praveen MC

## Objective
Build an automated news classification system capable of categorizing articles into predefined categories using traditional machine learning and transformer-based NLP techniques.

## Technologies Used
- Python
- Pandas
- NLTK
- Scikit-Learn
- TF-IDF
- DistilBERT
- Transformers

## Models Evaluated
- Multinomial Naive Bayes
- Logistic Regression
- Linear SVM
- Random Forest
- DistilBERT

## Results
Achieved high classification performance through extensive preprocessing, feature engineering, hyperparameter tuning, and transformer-based modeling.

# Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings('ignore')

# Listing Data

In [ ]:
df = pd.read_csv("data/flipitnews-data.csv")
df.head()

# Observations on Data

### Data Info

In [ ]:
df.info()

### Data Shape

In [ ]:
df.shape

### Column Names

In [ ]:
df.columns

In [ ]:
df.dtypes

### CHecking for Null Values

In [ ]:
df.isnull().sum()

### Checking for Duplicates

In [ ]:
df.duplicated().sum()

# Dataset Overview and Initial Observations

Before performing Exploratory Data Analysis (EDA) and building NLP models, it is important to thoroughly understand the structure, quality, and characteristics of the dataset. This initial inspection helps identify potential issues such as missing values, duplicate records, incorrect data types, and data inconsistencies that may affect downstream analysis and model performance.

---

### The dataset contains:

**2225 rows (news articles)**
**2 columns (features)**

This indicates that the dataset is moderately sized and suitable for NLP experimentation and model comparison.

---

The dataset consists of:

### Column	Description
**Category:**	The category/domain to which the news article belongs
**Article:**	The complete textual content of the news article

The dataset appears to be structured properly for a supervised machine learning classification problem.

## Column Names
The dataset contains two primary variables:

### 1. Category

This is the target variable that the model will predict. It contains labels such as:

Politics

Technology

Sports

Business

Entertainment

### 2. Article
This is the input feature containing raw textual news content.

This column will require extensive NLP preprocessing before model training.

## Data Type Analysis

Both columns are of datatype object, which is expected because:

Category contains categorical text labels

Article contains unstructured textual data

Since machine learning models cannot directly process raw text, the Article column must later be transformed into numerical representations using NLP techniques such as:

Bag of Words (BoW)

TF-IDF Vectorization

### Dataset Information
Key insights obtained:

#### Total Entries

The dataset contains all 2225 entries without missing records.

#### Memory Usage

The dataset consumes relatively low memory (~34.9 KB), indicating efficient handling during experimentation.

####  Non-Null Values

**Both columns contain complete information for every record.  No non-null values**

This is highly beneficial because:

No imputation techniques are required

Data preprocessing becomes simpler

Model training will not be interrupted by null values

### Missing Value Analysis

The dataset does not contain any missing values.

#### Business Significance

Having a complete dataset improves:

Data reliability

Consistency of analysis

Model performance stability

Since textual datasets often contain incomplete records, the absence of null values is a strong advantage for this project.

### Duplicate Value Analysis

**The dataset contains 99 duplicate records.**

Duplicate news articles may negatively impact the machine learning pipeline because:

The model may memorize repeated patterns

Evaluation metrics may become artificially inflated

Data leakage can occur between training and testing datasets

Certain categories may become overrepresented

Therefore, duplicate rows should be removed during preprocessing to maintain dataset integrity and ensure fair model evaluation.

###  Nature of the Problem

Based on the dataset structure, this is identified as a  **Supervised Multi-Class Text Classification Problem**, where:

| Component      | Description         |
|----------------|---------------------|
| Input Feature  | News Article Text   |
| Target Output  | News Category       |

The machine learning model will learn patterns in textual content and classify unseen articles into their appropriate categories.

## NLP Perspective of the Dataset

Unlike structured tabular datasets, textual data is unstructured and requires multiple preprocessing steps before modeling.

The **Article** column will undergo several NLP transformations such as:

Planned NLP Preprocessing Steps
###1. Text Cleaning
Removing punctuation
Removing special characters
Converting text to lowercase
###2. Stopword Removal

Removing common words such as the, is, and, of

which do not contribute significantly to meaning.

### 3. Tokenization

Breaking sentences into individual words/tokens.

### 4. Lemmatization

Reducing words to their root/base form.

Example:

| Original Word | Lemmatized Form |
|---------------|-----------------|
| running       | run             |
| studies       | study           |

### 5. Vectorization

Converting text into numerical format using:

Bag of Words
TF-IDF

These numerical representations will later be used for machine learning model training.

## Initial Business Understanding

The dataset aligns closely with the business objective of FlipItNews:

Automatically categorize articles

Improve content recommendation

Enhance user engagement

Deliver personalized news experiences

A successful NLP classification model can help the platform:

Organize large-scale news content efficiently

Improve search relevance

Increase reader retention

Deliver targeted content to users

## Key Takeaways from Initial Dataset Inspection
### Strengths of the Dataset

No missing values

Clear target labels

Well-structured format

Suitable size for NLP experimentation

Ready for preprocessing and modeling

### Issues Identified

Presence of duplicate records (99 duplicates)

Raw text requires extensive preprocessing

Text data must be converted into numerical form

## Conclusion

The initial dataset inspection reveals that the FlipItNews dataset is clean, structured, and highly suitable for an NLP-based text classification project.

The next stages of the analysis will focus on:

Duplicate removal

Category distribution analysis

Text preprocessing

Feature engineering

NLP model building

This foundational understanding is critical for designing an efficient and accurate news categorization system.


# Univariate Analysis

Univariate Analysis focuses on analyzing a single variable at a time to understand its distribution, characteristics, and patterns within the dataset.

In this NLP business case study, the primary focus of univariate analysis is:

1. Distribution of news article categories
2. Basic characteristics of textual articles
3. Article length distribution

This analysis helps us understand:
- Which categories dominate the dataset
- Whether the dataset is balanced
- Variations in article sizes
- Initial textual patterns before preprocessing

## Count of Articles in Each Category

In [ ]:
df['Category'].value_counts()

## Visualization of 'Category' Distribution

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    x=df['Category'],
    order=df['Category'].value_counts().index
)

plt.title('Distribution of News Categories')
plt.xlabel('Category')
plt.ylabel('Number of Articles')

plt.xticks(rotation=45)

plt.show()

# Category Distribution Analysis

Understanding the distribution of target classes is an important step in any classification problem because it helps identify whether the dataset is balanced or skewed toward certain categories.

In the FlipItNews dataset, the target variable is `Category`, which represents the domain to which each news article belongs.

---

# Category-wise Article Count

| Category | Number of Articles |
|---|---|
| Sports | 511 |
| Business | 510 |
| Politics | 417 |
| Technology | 401 |
| Entertainment | 386 |

---

# Visualization Insight

The count plot illustrates the distribution of news articles across all five categories.

## Key Observations

### 1. Sports Category Contains the Highest Number of Articles

The **Sports** category contains the maximum number of news articles with **511 articles**. This suggests that sports-related news content is highly represented in the dataset.

### 2. Business Category is Almost Equally Represented

The Business category contains **510 articles**, which is nearly identical to the Sports category.

This indicates a strong focus on business and finance-oriented news content within the platform.

3. Entertainment Category Contains the Lowest Number of Articles

The Entertainment category contains **386 articles** making it the least represented category in the dataset.

Models trained on this dataset may therefore have comparatively less learning exposure for entertainment-related content.

## Dataset Balance Observation

The dataset appears to be reasonably balanced overall because the difference between the largest and smallest categories is not extremely large.

**Approximate range: 511 - 386 = 125 articles**

This level of variation is generally acceptable for multi-class classification tasks.

## Machine Learning Perspective

A relatively balanced dataset is advantageous because:

Models are less likely to become biased toward majority classes.

Prediction performance across categories becomes more stable.

Evaluation metrics such as precision, recall, and F1-score become more reliable.

If severe imbalance existed, techniques such as, oversampling, undersampling, then class weighting might have been required.

However, in this case, the dataset distribution appears sufficiently balanced for initial model building.

## Business Perspective

The category distribution provides insights into FlipItNews' content strategy.

### Possible Business Interpretations
High representation of Sports and Business articles suggests that user engagement may be stronger in these domains.

Business-related dominance aligns with FlipItNews' broader focus on finance, investment, and economic awareness.

### Lower entertainment content may indicate:

lesser platform focus

lower publishing frequency

lower business priority in entertainment journalism

## NLP Perspective

Balanced category representation is particularly important in NLP classification because:

The model learns vocabulary patterns specific to each category.

Categories with fewer samples may suffer from:

weaker contextual learning

poorer generalization

lower recall scores

Since all categories have several hundred articles, the dataset is suitable for:

TF-IDF vectorization

Bag of Words modeling

Traditional machine learning classifiers

Advanced NLP models

## Overall Conclusion

The category distribution analysis indicates that:

The dataset is reasonably balanced

Sports and Business dominate the dataset

Entertainment is the least represented category

No severe class imbalance issue is observed

The dataset is suitable for multi-class NLP classification tasks

This balanced distribution is expected to help machine learning models perform more consistently across categories.

# Article Length Analysis

Analyzing article length is an important step in NLP because the size of textual content can significantly influence:

- Model performance
- Computational complexity
- Vocabulary richness
- Contextual understanding

Longer articles generally provide richer semantic information, while shorter articles may contain limited contextual details.

To better understand the textual characteristics of the FlipItNews dataset, article length distribution was analyzed.

---

# Creating Article Length Feature

The length of each article was calculated using the number of characters present in the text.

# Creating 'Article Length' Feature

In [ ]:
df['Article_Length'] = df['Article'].apply(len)

# Statistical Summary of Article Length

In [ ]:
df['Article_Length'].describe()

# Visualization of Article Length Distribution

In [ ]:
plt.figure(figsize=(10,5))

sns.histplot(df['Article_Length'], bins=30, kde=True)

plt.title('Distribution of Article Length')
plt.xlabel('Article Length')
plt.ylabel('Frequency')

plt.show()

# Category-wise Article Length Boxplot

In [ ]:
plt.figure(figsize=(10,6))

sns.boxplot(
    x='Category',
    y='Article_Length',
    data=df
)

plt.title('Article Length by Category')

plt.xticks(rotation=45)

plt.show()

# Category-wise Article Length Analysis

To further understand the textual characteristics of the dataset, a boxplot was created to analyze article length distribution across different news categories.

This visualization helps identify:

- Differences in article length across categories
- Variability in writing style
- Presence of outliers
- Distribution spread within each category

---

# Interpretation of Boxplot

## 1. Politics Articles Tend to be Longer

The **Politics** category shows comparatively higher median article lengths and wider spread.

This suggests that political news articles are generally:
- More detailed
- Analytical in nature
- Rich in contextual information

Additionally, the Politics category contains some of the largest outliers in the dataset, including articles exceeding 25000 characters.


This may represent:

Editorial analysis

Investigative journalism

Long-form political reporting
##2. Technology Articles Also Show High Variability

The Technology category contains:

Moderate-to-high median article lengths

Several extreme outliers

This indicates that technology articles may include:

Product reviews

Technical analysis

Detailed innovation reports

The category demonstrates significant variability in content depth.

## 3. Business and Entertainment Articles are Comparatively Compact

The Business and Entertainment categories show:

Lower median article lengths
Relatively tighter distributions

This suggests that these articles are generally:

More concise

More standardized in writing style

Less variable in length

## 4. Sports Articles Show Moderate Consistency

The Sports category demonstrates:

Moderate article lengths

Comparatively narrower interquartile range

This indicates more consistency in sports reporting compared to categories like Politics or Technology.

Sports news articles may follow more structured reporting formats such as:

Match summaries

Score updates

Event highlights

##Presence of Outliers

All categories contain outliers, indicating the presence of unusually long articles.

However, the outlier density is especially noticeable in:

Politics

Technology

Entertainment

These outliers are not necessarily errors because long-form journalism is common in several domains.

## NLP Perspective

Variation in article length across categories is important in NLP because:

Longer articles provide richer vocabulary and context.

Categories with detailed reporting may generate:

larger feature spaces

higher dimensionality

increased computational cost

The observed variability suggests that:

Preprocessing
Vectorization

Feature engineering

will play a critical role in building efficient classification models.

## Business Perspective

The differences in article length reflect the nature of reporting across various news domains.

| Category | Likely Content Style |
|---|---|
| Politics | Analytical and detailed |
| Technology | Technical and explanatory |
| Business | Structured and concise |
| Sports | Consistent event reporting |
| Entertainment | Mixed and variable |

### These insights can help FlipItNews better understand:


Reader engagement patterns

Content consumption behavior

Category-specific content strategies

## Overall Conclusion

The category-wise article length analysis reveals that:

Politics and Technology articles tend to be longer and more variable

Business and Sports articles are relatively compact and consistent

Multiple categories contain long-text outliers

Writing style and content depth vary significantly across domains

Textual variability will influence NLP preprocessing and model behavior

This analysis provides deeper insight into the structure and complexity of the textual dataset before moving into NLP preprocessing.

In [ ]:
df['Article'][0]

In [ ]:
df.sample(3)

## Removing Duplicates

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.duplicated().sum()

In [ ]:
df.shape

In [ ]:
(df['Category'].value_counts(normalize=True)*100).round(2)

# Text Preprocessing/NLP Cleaning

## 1. Lowercase Conversion

In [ ]:
df['Article_lower'] = df['Article'].str.lower()

In [ ]:
df[['Article','Article_lower']].head(2)

# Lowercase Conversion

The first step in NLP preprocessing is converting all text into lowercase format.

## Why Lowercasing is Important

In textual data, words with different cases are treated as completely different tokens by machine learning algorithms.

Example:

- "News"
- "news"
- "NEWS"

would all be interpreted as separate words.

This leads to:
- unnecessary increase in vocabulary size
- sparse feature representation
- reduced model efficiency

Lowercasing standardizes the textual data by converting all characters into lowercase form.

---

# Benefits of Lowercasing

- Reduces vocabulary dimensionality
- Improves consistency in text representation
- Enhances word frequency analysis
- Improves vectorization efficiency
- Helps machine learning models generalize better

---

# Implementation

A new column named `Article_lower` was created to store the lowercase version of the original article text while preserving the raw data for reference.


## 2. Remove Special Characters & Punctuation

In [ ]:
df['Article_clean'] = df['Article_lower'].apply(
    lambda x: re.sub(r'[^a-z\s]', '', x)
)

In [ ]:
df[['Article_lower','Article_clean']].head(2)

# Removing Special Characters and Punctuation

After converting text to lowercase, the next preprocessing step involves removing special characters, punctuation marks, and numerical values from the textual data.

---

# Why This Step is Important

Raw textual data often contains:
- punctuation marks
- symbols
- numbers
- unwanted special characters

Examples include:  .,!?;:-_()[]{}@#$%^&*

## Regex-Based Cleaning

In [ ]:
df['Article_clean'] = df['Article_lower'].apply(
    lambda x: re.sub(r'[^a-zA-Z\s]', ' ', x)
)

In [ ]:
df['Article_clean'] = df['Article_clean'].apply(
    lambda x: re.sub(r'\s+', ' ', x).strip()
)

In [ ]:
df['Article_clean'][0]

## 3. Tokenization

In [ ]:
nltk.download('punkt_tab')

In [ ]:
df['Article_tokens'] = df['Article_clean'].apply(word_tokenize)

In [ ]:
df[['Article_clean','Article_tokens']].head(2)

# Tokenization

After cleaning the textual data, the next NLP preprocessing step is tokenization.

---

# What is Tokenization?

Tokenization is the process of breaking textual data into smaller meaningful units called tokens.

These tokens are usually:
- words
- subwords
- sentences

In this project, word-level tokenization is performed.

---

## 4. Stopword Removal

In [ ]:
# =========================
# Stopword Removal
# =========================

# Store English stopwords
stop_words = set(stopwords.words('english'))

# Remove stopwords from tokenized text
df['Article_no_stopwords'] = df['Article_tokens'].apply(
    lambda tokens: [word for word in tokens if word not in stop_words]
)

# Verify output
df[['Article_tokens', 'Article_no_stopwords']].head(2)

# Stopword Removal

Stopwords are commonly occurring words in a language that generally carry very little meaningful information for NLP tasks.

Examples include:

```text
the, is, in, of, and, to

## 5. Lemmatization

In [ ]:
# =========================
# Lemmatization
# =========================

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

# Apply lemmatization
df['Article_lemmatized'] = df['Article_no_stopwords'].apply(
    lambda tokens: [lemmatizer.lemmatize(word) for word in tokens]
)

# Verify output
df[['Article_no_stopwords', 'Article_lemmatized']].head(2)

# Lemmatization

After removing stopwords, the next NLP preprocessing step is lemmatization.

Lemmatization is the process of converting words into their base or dictionary form, known as the lemma.

---

# Examples

| Original Word | Lemmatized Form |
|---|---|
| viewers | viewer |
| books | book |
| systems | system |
| technologies | technology |

---

# Why Lemmatization is Important

Different forms of a word often carry the same semantic meaning.

For example:

```text
connect, connected, connecting, connection

## 6. Clearned Text Creation

In [ ]:
# =========================
# Create Final Clean Text
# =========================

df['Final_Text'] = df['Article_lemmatized'].apply(
    lambda tokens: ' '.join(tokens)
)

# Verify output
df[['Article','Final_Text']].head(2)

# Creating Final Clean Text

After completing all major preprocessing steps such as:
- lowercase conversion
- special character removal
- tokenization
- stopword removal
- lemmatization

the processed text existed in the form of tokenized word lists.

However, text vectorization techniques such as:
- Bag of Words (BoW)
- TF-IDF

require textual data in sentence/string format rather than list format.

Therefore, the cleaned tokens were combined back into complete sentences to create the final preprocessed text column.

---

# Post-Preprocessing Text Analysis

## 1. Flatten Tokens and Count Frequencies

In [ ]:
# =========================
# Most Frequent Words
# =========================

from collections import Counter

# Flatten all tokens into one list
all_words = []

for tokens in df['Article_lemmatized']:
    all_words.extend(tokens)

# Count word frequencies
word_freq = Counter(all_words)

# Convert to DataFrame
freq_df = pd.DataFrame(
    word_freq.items(),
    columns=['Word', 'Frequency']
)

# Sort by frequency
freq_df = freq_df.sort_values(
    by='Frequency',
    ascending=False
)

# Display top 10 words
freq_df.head(10)

## Visualization of Top Words

In [ ]:
# Top 10 frequent words

top_words = freq_df.head(10)

plt.figure(figsize=(10,5))

sns.barplot(
    x='Frequency',
    y='Word',
    data=top_words
)

plt.title('Top 10 Most Frequent Words')

plt.show()

# Most Frequent Words Analysis

After completing text preprocessing, word frequency analysis was performed to identify the most commonly occurring words across all news articles.

This analysis helps understand:
- dominant vocabulary
- recurring themes
- contextual focus areas
- common discussion patterns within the dataset

---

# Top 10 Most Frequent Words

| Word | Frequency |
|---|---|
| said | 6928 |
| year | 3193 |
| mr | 2900 |
| would | 2472 |
| also | 2035 |
| u | 1938 |
| people | 1896 |
| new | 1877 |
| one | 1851 |
| time | 1569 |

---

# Interpretation of Findings

## 1. "said" is the Most Frequent Word

The word:

```text
said

## 2. Word Cloud Generation

In [ ]:
from wordcloud import WordCloud

In [ ]:
text = ' '.join(df['Final_Text'])

In [ ]:
# Generate word cloud

wordcloud = WordCloud(
    width=1000,
    height=500,
    background_color='white'
).generate(text)

# Plot word cloud

plt.figure(figsize=(15,8))

plt.imshow(wordcloud, interpolation='bilinear')

plt.axis('off')

plt.title('Word Cloud of News Articles')

plt.show()

# Word Cloud Analysis

A Word Cloud was generated using the fully preprocessed textual data to visually represent the most frequently occurring words across all news articles.

In a word cloud:
- larger words indicate higher frequency
- smaller words indicate lower frequency

This visualization provides a quick and intuitive understanding of dominant themes and vocabulary present in the dataset.

---

# Observation from the Word Cloud

Several highly prominent words can be observed, including:

- said
- new
- company
- people
- game
- government
- market
- film
- player
- sale
- technology-related terms

---

# Interpretation of Findings

## 1. Strong Presence of News Reporting Vocabulary

Words such as:

```text
said, people, government

# Category-wise Frequent Words

In [ ]:
# =========================
# Category-wise Frequent Words
# =========================

from collections import Counter

# Get unique categories
categories = df['Category'].unique()

# Loop through categories
for category in categories:

    print(f"\nTop words in {category} category:\n")

    # Filter category
    category_text = df[df['Category'] == category]

    # Combine all words
    words = []

    for tokens in category_text['Article_lemmatized']:
        words.extend(tokens)

    # Count frequencies
    word_freq = Counter(words)

    # Display top 10 words
    print(word_freq.most_common(10))

# Category-wise Frequent Words Analysis

To better understand the linguistic patterns and dominant themes within each news category, category-wise word frequency analysis was performed on the fully preprocessed text.

This analysis helps identify:
- category-specific vocabulary
- dominant discussion topics
- contextual differences between domains
- important keywords that may influence classification models

---

# Technology Category

### Top Frequent Words

| Word | Frequency |
|---|---|
| said | 1368 |
| people | 829 |
| game | 705 |
| technology | 546 |
| mobile | 535 |
| one | 489 |
| phone | 486 |
| year | 466 |
| also | 460 |
| mr | 452 |

---

## Interpretation

The Technology category contains several domain-specific terms such as:
- technology
- mobile
- phone
- game

This indicates strong coverage of:
- consumer electronics
- mobile devices
- gaming technology
- digital innovation

The presence of the word:
*game* suggests overlap between:
gaming technology
entertainment technology
digital media

# Business Category

### Top Frequent Words

| Word | Frequency |
|---|---|
| said | 1655 |
| year | 943 |
| u | 805 |
| bn | 782 |
| company | 624 |
| mr | 599 |
| firm | 533 |
| market | 521 |
| would | 460 |
| also | 432 |

---

## Interpretation

The Business category strongly reflects:
- financial reporting
- corporate discussions
- economic analysis

Important business-oriented words include:
- company
- firm
- market
- bn

The token text *bn* likely refers to *billion*, which is frequently used in financial journalism.

This indicates strong coverage of:

market performance
company activities
business growth
economic developments

The vocabulary suggests that the dataset contains substantial business and finance-related reporting.


---

# Sports Category

### Top Frequent Words

| Word | Frequency |
|---|---|
| said | 928 |
| year | 667 |
| game | 643 |
| time | 481 |
| first | 478 |
| player | 474 |
| win | 463 |
| england | 457 |
| back | 398 |
| would | 396 |

---

## Interpretation

The Sports category clearly reflects:
- match reporting
- player discussions
- tournament coverage
- sports event analysis

Key sports-related terms include:
- game
- player
- win
- england

The frequent occurrence of text *england* suggests strong coverage of:

UK sports events
football
cricket
international competitions

The vocabulary demonstrates that the Sports category primarily focuses on:

competitive events
player performances
match outcomes
sports journalism narratives


---

# Entertainment Category

### Top Frequent Words

| Word | Frequency |
|---|---|
| film | 926 |
| said | 803 |
| year | 653 |
| best | 589 |
| award | 507 |
| music | 423 |
| u | 396 |
| show | 395 |
| also | 382 |
| one | 372 |

---

## Interpretation

The Entertainment category prominently features words related to:
- cinema
- music
- television
- award events

Important entertainment-specific terms include:
- film
- award
- music
- show

This indicates strong focus on:
- movie industry reporting
- entertainment programs
- celebrity news
- cultural events

The vocabulary reflects a media-oriented and audience-centric reporting style commonly associated with entertainment journalism.

---

# Politics Category

### Top Frequent Words

| Word | Frequency |
|---|---|
| said | 2174 |
| mr | 1646 |
| would | 1008 |
| labour | 735 |
| government | 728 |
| party | 691 |
| election | 636 |
| people | 607 |
| minister | 544 |
| blair | 542 |

---

## Interpretation

The Politics category strongly reflects:
- political discussions
- governance
- elections
- policy debates

Important political terms include:
- government
- party
- election
- minister
- labour

The frequent occurrence of *blair* suggests that the dataset contains substantial coverage of:

UK politics
parliamentary discussions
Tony Blair-era political events

The vocabulary highlights strong emphasis on:

government decisions
political leadership
party activities
electoral processes

This category exhibits a highly formal and debate-oriented reporting style.

# Category-wise Word Clouds

In [ ]:
# =========================
# Category-wise Word Clouds
# =========================

# Create figure
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Flatten axes for easy looping
axes = axes.flatten()

# Get unique categories
categories = df['Category'].unique()

# Generate word clouds
for i, category in enumerate(categories):

    # Combine text for that category
    text = ' '.join(
        df[df['Category'] == category]['Final_Text']
    )

    # Generate word cloud
    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color='white'
    ).generate(text)

    # Plot
    axes[i].imshow(wordcloud, interpolation='bilinear')

    axes[i].set_title(f'{category} Word Cloud')

    axes[i].axis('off')

# Remove extra empty subplot
fig.delaxes(axes[5])

plt.tight_layout()

plt.show()

# N-gram Analysis

## 2 Bigram Analysis

In [ ]:
# =========================
# Bigram Analysis
# =========================

from sklearn.feature_extraction.text import CountVectorizer

# Initialize CountVectorizer for bigrams
bigram_vectorizer = CountVectorizer(ngram_range=(2,2))

# Fit and transform text
bigram_matrix = bigram_vectorizer.fit_transform(df['Final_Text'])

# Sum frequencies
bigram_counts = bigram_matrix.sum(axis=0)

# Create frequency dictionary
bigram_freq = [
    (word, bigram_counts[0, idx])
    for word, idx in bigram_vectorizer.vocabulary_.items()
]

# Convert to DataFrame
bigram_df = pd.DataFrame(
    bigram_freq,
    columns=['Bigram', 'Frequency']
)

# Sort frequencies
bigram_df = bigram_df.sort_values(
    by='Frequency',
    ascending=False
)

# Display top 10 bigrams
bigram_df.head(10)

## Visualization of Bigram Analysis

In [ ]:
# Top 10 Bigrams

top_bigrams = bigram_df.head(10)

plt.figure(figsize=(10,5))

sns.barplot(
    x='Frequency',
    y='Bigram',
    data=top_bigrams
)

plt.title('Top 10 Most Frequent Bigrams')

plt.show()

# Bigram Analysis

Bigram analysis was performed to identify the most frequently occurring two-word combinations within the preprocessed news articles.

Unlike unigram analysis, bigrams preserve word order and contextual relationships, providing deeper semantic insights into the dataset.

This helps identify:
- commonly discussed topics
- repeated contextual phrases
- domain-specific expressions
- important linguistic patterns

---

# Top 10 Most Frequent Bigrams

| Bigram | Frequency |
|---|---|
| last year | 480 |
| told bbc | 344 |
| said mr | 339 |
| year old | 337 |
| mr blair | 312 |
| prime minister | 304 |
| mr brown | 236 |
| chief executive | 199 |
| said would | 182 |
| tony blair | 176 |

---

# Interpretation of Findings

## 1. Strong Presence of News Reporting Language

Several bigrams such as:
- told bbc
- said mr
- said would

reflect standard journalistic reporting patterns commonly found in news articles.

This confirms that the dataset contains authentic news-reporting style content involving:
- interviews
- official statements
- quotations
- media reporting

---

## 2. Political Context is Highly Dominant

Important political bigrams include:
- prime minister
- mr blair
- tony blair
- mr brown

These phrases strongly suggest that the dataset contains substantial coverage of:
- UK politics
- political leadership
- parliamentary discussions
- government affairs

The repeated occurrence of *tony blair* indicates significant discussion surrounding former UK Prime Minister Tony Blair.

## 3. Business and Corporate Reporting Patterns

The bigram *chief executive* reflects corporate and business-related reporting.

This suggests coverage involving:

company leadership
executive decisions
financial organizations
corporate announcements

## 4. Temporal and Descriptive Phrases

Bigrams such as *last year*, *year old* indicate that many articles contain:

time references
historical comparisons
age-related descriptions
event timelines

These are commonly observed in journalistic narratives.

# NLP Perspective

Bigram analysis provides significantly richer contextual information than unigram analysis because:

it preserves word relationships
captures semantic phrases
improves contextual understanding

For example:

Unigram	- Limited Meaning
prime	- ambiguous
minister	- ambiguous
But together *prime minister* clearly represents a political concept.

This contextual understanding becomes highly valuable for:

feature engineering
TF-IDF modeling
topic detection
text classification

# Business Perspective

The bigram analysis highlights several dominant themes across the FlipItNews dataset, including:

political leadership
media reporting
corporate management
event timelines

## These insights help identify:

major discussion areas
frequently occurring public figures
common reporting structures
domain-specific content patterns

Such information can support:

content categorization
recommendation systems
news trend analysis
personalized news delivery

# Overall Conclusion

The bigram analysis demonstrates that the dataset contains rich contextual and domain-specific language patterns.

The presence of meaningful phrases such as:

prime minister
chief executive
tony blair

indicates that preprocessing has successfully preserved important semantic relationships within the text.

These contextual features are highly valuable for:

NLP feature engineering
TF-IDF vectorization
machine learning classification models

## 3. Trigram Frequency Analysis

In [ ]:
# =========================
# Trigram Analysis
# =========================

# Initialize CountVectorizer for trigrams
trigram_vectorizer = CountVectorizer(ngram_range=(3,3))

# Fit and transform text
trigram_matrix = trigram_vectorizer.fit_transform(df['Final_Text'])

# Sum frequencies
trigram_counts = trigram_matrix.sum(axis=0)

# Create frequency dictionary
trigram_freq = [
    (word, trigram_counts[0, idx])
    for word, idx in trigram_vectorizer.vocabulary_.items()
]

# Convert to DataFrame
trigram_df = pd.DataFrame(
    trigram_freq,
    columns=['Trigram', 'Frequency']
)

# Sort frequencies
trigram_df = trigram_df.sort_values(
    by='Frequency',
    ascending=False
)

# Display top 10 trigrams
trigram_df.head(10)

## Visualization of Trigram Frequency Analysis

In [ ]:
# Top 10 Trigrams

top_trigrams = trigram_df.head(10)

plt.figure(figsize=(10,5))

sns.barplot(
    x='Frequency',
    y='Trigram',
    data=top_trigrams
)

plt.title('Top 10 Most Frequent Trigrams')

plt.show()

# Trigram Analysis

Trigram analysis was performed to identify the most frequently occurring three-word combinations within the preprocessed news articles.

Unlike unigrams and bigrams, trigrams capture more complete semantic and contextual information by preserving longer phrase structures.

This helps identify:
- named entities
- official titles
- recurring media phrases
- domain-specific expressions
- contextual reporting patterns

---

# Top 10 Most Frequent Trigrams

| Trigram | Frequency |
|---|---|
| told bbc news | 136 |
| bbc news website | 86 |
| told bbc radio | 74 |
| mr kilroy silk | 57 |
| leader michael howard | 57 |
| told bbc sport | 48 |
| mr blair said | 46 |
| million dollar baby | 46 |
| radio today programme | 46 |
| bbc radio today | 44 |

---

# Interpretation of Findings

## 1. Strong BBC Media Reporting Presence

Several trigrams such as:
- told bbc news
- bbc news website
- told bbc radio
- told bbc sport

indicate that the dataset contains substantial BBC-related news reporting content.

This suggests:
- syndicated journalism
- formal media citations
- interview-based reporting
- official news broadcasting references

---

## 2. Political Context is Clearly Visible

Trigrams such as:
- mr blair said
- leader michael howard
- mr kilroy silk

reflect strong political discussion and UK political coverage within the dataset.

These phrases indicate:
- parliamentary discussions
- party leadership coverage
- public political statements
- election-related reporting

---

## 3. Entertainment and Media References

The trigram *million dollar baby*

refers to a well-known film title, confirming the presence of:

entertainment journalism
movie discussions
media reporting

This demonstrates that trigram analysis can successfully capture complete named entities and titles.

## 4. Structured Reporting Language

Several trigrams involve:

reporting organizations
broadcast programs
official statements

Examples include:

radio today programme
bbc radio today

These phrases reflect highly structured journalistic writing styles.

# NLP Perspective

Trigram analysis provides deeper contextual understanding than unigram and bigram analysis because it captures:

longer semantic relationships
named entities
complete contextual phrases

For example:

Phrase Level	Context Clarity
blair	vague
mr blair	clearer
mr blair said	highly contextual

This richer contextual information becomes highly valuable for:

NLP feature engineering
contextual classification
topic modeling
named entity understanding
semantic analysis

# Business Perspective

The trigram analysis reveals that the FlipItNews dataset contains:

formal news reporting structures
political commentary
broadcast journalism references
entertainment-related content

The dominance of BBC-related phrases suggests strong representation of:

UK-centric journalism
broadcast news sources
media-driven reporting

These insights help:

understand content structure
identify dominant reporting styles
improve contextual recommendation systems
support category-specific content analysis

# Overall Conclusion

The trigram analysis demonstrates that the dataset contains rich contextual and domain-specific phrase structures.

The presence of meaningful trigrams such as:

told bbc news
mr blair said
million dollar baby

confirms that preprocessing has successfully preserved higher-level semantic relationships within the textual data.

These contextual phrase patterns are highly valuable for:

advanced NLP analysis
feature engineering
TF-IDF modeling
supervised text classification systems

# Feature Engineering

## Bag of Words (BoW)
#### Bag of Words converts textual data into numerical feature vectors by counting how many times each word appears in a document. It ignores grammar, word order, sentence structure and focuses only on word frequency

In [ ]:
# =========================
# Bag of Words Vectorization
# =========================

from sklearn.feature_extraction.text import CountVectorizer

# Initialize CountVectorizer
bow_vectorizer = CountVectorizer()

# Fit and transform the text
X_bow = bow_vectorizer.fit_transform(df['Final_Text'])

# Target variable
y = df['Category']

# Shape of BoW matrix
X_bow.shape

In [ ]:
len(bow_vectorizer.vocabulary_)

In [ ]:
# First 20 vocabulary words

list(bow_vectorizer.vocabulary_.keys())[:20]

In [ ]:
X_bow

# Results of Bag of Words (BoW) Vectorization

The Bag of Words (BoW) technique successfully transformed the preprocessed textual news articles into structured numerical feature vectors suitable for machine learning algorithms.

---

# Key Results

- Total number of documents/articles:
  - 2,126

- Total vocabulary size:
  - 24,728 unique words

- Sparse matrix representation generated:
  - `(2126, 24728)`

- Non-zero stored elements:
  - 313,824

---

# Key Observations

- The dataset contains a rich and diverse vocabulary across multiple news categories.
- Each article is represented numerically based on word occurrence frequency.
- The generated feature matrix is highly sparse because each article contains only a small subset of the total vocabulary.
- Sparse matrix representation efficiently handles high-dimensional NLP data while reducing memory usage.

---

# Importance of BoW

Bag of Words provides a foundational text representation technique that enables:
- text classification
- machine learning modeling
- feature engineering
- predictive NLP analysis

The BoW representation now forms the basis for:
- training classification models
- comparing vectorization techniques
- evaluating NLP model performance

---

# TF-IDF Vectorization
#### TF-IDF stands for **Term Frequency – Inverse Document Frequency**

#### Unlike Bag of Words, TF-IDF not only counts word frequency, but also measures how important a word is within a document relative to the entire dataset.

#### Key idea is words appearing very frequently in **all documents get lower importance.**  Words appearing frequently in **specific documents categories get higher importance**

In [ ]:
# =========================
# TF-IDF Vectorization
# =========================

from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer()

# Fit and transform the text
X_tfidf = tfidf_vectorizer.fit_transform(df['Final_Text'])

# Target variable
y = df['Category']

# Shape of TF-IDF matrix
X_tfidf.shape

In [ ]:
# Vocabulary size

len(tfidf_vectorizer.vocabulary_)

In [ ]:
# First 20 vocabulary words

list(tfidf_vectorizer.vocabulary_.keys())[:20]

In [ ]:
X_tfidf

# Results of TF-IDF Vectorization

The TF-IDF (Term Frequency–Inverse Document Frequency) technique successfully transformed the preprocessed news articles into weighted numerical feature vectors suitable for machine learning algorithms.

Unlike Bag of Words (BoW), TF-IDF assigns importance scores to words based on:
- their frequency within a document
- and their uniqueness across the entire dataset

---

# Key Results

- Total number of documents/articles:
  - 2,126

- Total vocabulary size:
  - 24,728 unique words

- TF-IDF matrix shape:
  - `(2126, 24728)`

- Non-zero stored elements:
  - 313,824

- Sparse matrix type:
  - Compressed Sparse Row (CSR) Matrix

---

# Sample Vocabulary Terms

['tv',
 'future',
 'hand',
 'viewer',
 'home',
 'theatre',
 'system',
 'plasma',
 'high',
 'definition',
 'digital',
 'video',
 'recorder',
 'moving',
 'living',
 'room',
 'way',
 'people',
 'watch',
 'radically']

These vocabulary terms confirm that preprocessing has successfully:

cleaned the textual data
preserved meaningful words
standardized vocabulary through lemmatization

##Key Observations
TF-IDF preserves the same vocabulary size as Bag of Words but applies weighted importance scores instead of raw counts.
Frequently occurring common words receive lower weights.
More category-specific and informative words receive higher weights.
The resulting feature matrix remains highly sparse because each article contains only a small subset of the total vocabulary.

##Importance of TF-IDF

TF-IDF improves NLP feature representation by:

reducing the dominance of common words
emphasizing discriminative vocabulary
improving category separability
enhancing machine learning model performance

This makes TF-IDF particularly useful for:

text classification
topic detection
recommendation systems
semantic analysis

##Overall Conclusion

The TF-IDF vectorization successfully converted the news articles into weighted numerical representations while preserving meaningful contextual vocabulary.

The generated sparse feature matrix now forms a strong foundation for:

NLP classification models
predictive analytics
feature engineering
supervised machine learning algorithms

TF-IDF is expected to provide more informative and discriminative features compared to traditional Bag of Words representation.

In [ ]:
# Get feature names

feature_names = tfidf_vectorizer.get_feature_names_out()

feature_names[:20]

In [ ]:
# Calculate average TF-IDF score for each feature

mean_tfidf = X_tfidf.mean(axis=0).A1

# Create dataframe
tfidf_df = pd.DataFrame({
    'Word': feature_names,
    'TFIDF_Score': mean_tfidf
})

# Sort descending
tfidf_df = tfidf_df.sort_values(
    by='TFIDF_Score',
    ascending=False
)

# Display top 10 words
tfidf_df.head(10)

In [ ]:
# Top 10 TF-IDF Features

top_tfidf = tfidf_df.head(10)

plt.figure(figsize=(10,5))

sns.barplot(
    x='TFIDF_Score',
    y='Word',
    data=top_tfidf
)

plt.title('Top 10 TF-IDF Important Words')

plt.show()

# TF-IDF Feature Inspection

After applying TF-IDF vectorization, the average TF-IDF scores for all vocabulary words were calculated to identify the most important and influential terms across the news articles.

Unlike Bag of Words, TF-IDF assigns weights based on:
- word frequency within a document
- and uniqueness across the entire dataset

This helps identify words that carry stronger contextual importance.

---

# Top 10 TF-IDF Important Words

| Word | TF-IDF Score |
|---|---|
| said | 0.037972 |
| mr | 0.025615 |
| year | 0.022470 |
| would | 0.018399 |
| game | 0.016942 |
| film | 0.016660 |
| people | 0.015833 |
| new | 0.015337 |
| also | 0.015035 |
| one | 0.014287 |

---

# Interpretation of Findings

## 1. Reporting-Oriented Vocabulary Dominates

Words such as:
- said
- mr
- would

remain highly important because they appear frequently within journalistic reporting structures.

This reflects the formal nature of news-writing styles involving:
- quotations
- interviews
- public statements
- reported speech

---

## 2. Domain-Specific Terms are Emerging

Words such as:
- game
- film

carry stronger contextual importance because they are associated with specific news categories such as:
- Sports
- Entertainment

This demonstrates TF-IDF’s ability to capture category-relevant vocabulary.

---

## 3. Temporal and General Discussion Terms

Words such as:
- year
- new
- people

indicate that many articles discuss:
- recent developments
- societal topics
- ongoing events
- evolving trends

These are naturally common across news datasets.

---

# NLP Perspective

The TF-IDF feature inspection demonstrates that:
- words are no longer treated equally
- important contextual words receive higher weights
- category-relevant vocabulary becomes more influential

Compared to Bag of Words:
- TF-IDF reduces the impact of extremely common words
- and improves the importance of discriminative vocabulary

This generally improves:
- text classification
- semantic understanding
- machine learning performance

---

# Business Perspective

The important TF-IDF terms indicate that the FlipItNews dataset strongly revolves around:
- reporting and public communication
- sports discussions
- entertainment coverage
- event-driven narratives

These weighted features help improve:
- article categorization
- recommendation engines
- topic identification
- content personalization

---

# Overall Conclusion

The TF-IDF feature inspection confirms that the vectorization process has successfully identified influential and contextually meaningful vocabulary within the dataset.

The generated weighted feature representation is now highly suitable for:
- supervised machine learning models
- NLP classification tasks
- semantic analysis
- predictive text analytics

TF-IDF is expected to provide more informative features than traditional Bag of Words representation during model training.

---

# Model Buliding

## Model-1 - Naive Bayes: NLP Model Building using TF-IDF Features

In [ ]:
# =========================
# Train-Test Split
# =========================

from sklearn.model_selection import train_test_split

# Split TF-IDF features
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Check shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
stratify=y

## Baseline Model: Multinomial Naive Bayes

In [ ]:
# =========================
# Multinomial Naive Bayes
# =========================

from sklearn.naive_bayes import MultinomialNB

# Initialize model
nb_model = MultinomialNB()

# Train model
nb_model.fit(X_train, y_train)

# Predictions
y_pred_nb = nb_model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

# Accuracy
nb_accuracy = accuracy_score(y_test, y_pred_nb)

print("Naive Bayes Accuracy:", nb_accuracy)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_nb))

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_nb)

cm

In [ ]:
plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=nb_model.classes_,
    yticklabels=nb_model.classes_
)

plt.xlabel('Predicted')

plt.ylabel('Actual')

plt.title('Naive Bayes Confusion Matrix')

plt.show()

# Multinomial Naive Bayes Model Evaluation

A Multinomial Naive Bayes classifier was trained using TF-IDF vectorized textual features to classify news articles into their respective categories.

Naive Bayes is one of the most widely used baseline algorithms for NLP classification tasks because of its:
- simplicity
- computational efficiency
- strong performance on sparse textual data

---

# Model Performance

## Accuracy Score

python id="nb_acc
Naive Bayes Accuracy: 0.9601

The model achieved an overall accuracy of approximately:

96.0%

This indicates excellent classification performance on unseen news articles.

## Classification Report
Category	Precision	Recall	F1-Score
Business	0.91	1.00	0.95
Entertainment	1.00	0.89	0.94
Politics	0.95	0.98	0.96
Sports	0.97	1.00	0.99
Technology	1.00	0.90	0.95

## Interpretation of Results
1. Excellent Overall Performance

The model demonstrates very strong performance across all news categories with:

high precision
high recall
high F1-scores

This indicates that:

TF-IDF features effectively captured category-specific vocabulary
the preprocessing pipeline successfully reduced textual noise
categories are well separable using textual patterns
2. Sports Category Achieved Best Performance

The Sports category achieved:

Precision: 0.97
Recall: 1.00
F1-score: 0.99

This suggests that sports articles contain:

highly distinctive vocabulary
strong contextual consistency
easily identifiable textual patterns

Words such as:

game
player
win
england

likely contributed strongly to classification performance.

3. Business and Politics Also Performed Strongly

Both categories achieved excellent scores due to the presence of highly domain-specific terms such as:

market
company
government
election
minister

These discriminative terms help the model clearly distinguish between categories.

4. Slight Confusion in Entertainment and Technology

Some minor misclassifications were observed in:

Entertainment
Technology

This may occur because:

both categories sometimes share overlapping media-related vocabulary
technology news may contain entertainment-related content such as gaming or digital media

However, performance remains very strong overall.

## Confusion Matrix Insights

The confusion matrix shows that:

most articles were correctly classified
misclassifications are minimal
category separation is highly effective

Examples:

Business articles were classified almost perfectly
Sports articles achieved near-perfect classification
Technology articles occasionally overlapped with Business or Sports
## NLP Perspective

The strong performance of Naive Bayes confirms that:

TF-IDF vectorization is highly effective for text classification
sparse textual representations work well with probabilistic classifiers
preprocessing significantly improved feature quality

This demonstrates the effectiveness of:

tokenization
stopword removal
lemmatization
TF-IDF weighting

in building high-quality NLP pipelines.

Business Perspective

The high classification accuracy indicates that FlipItNews can effectively:

automate news categorization
improve recommendation systems
enhance search relevance
personalize user feeds
organize large volumes of textual content

Accurate automated classification reduces manual tagging effort and improves platform scalability.

## Overall Conclusion

The Multinomial Naive Bayes model achieved excellent performance using TF-IDF features, demonstrating that the textual data contains strong category-specific linguistic patterns.

The model successfully classified news articles with high accuracy and minimal confusion across categories.

This establishes a strong baseline for further:

hyperparameter tuning
advanced model development
comparative NLP modeling

## Hyperparameter Tuning Using GridSearchCV

In [ ]:
# =========================
# Hyperparameter Tuning
# Multinomial Naive Bayes
# =========================

from sklearn.model_selection import GridSearchCV

# Parameter grid
param_grid_nb = {
    'alpha': [0.01, 0.1, 0.5, 1.0, 2.0]
}

# GridSearchCV
grid_nb = GridSearchCV(
    MultinomialNB(),
    param_grid_nb,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

# Fit grid search
grid_nb.fit(X_train, y_train)

In [ ]:
# Best parameters

print("Best Parameters:", grid_nb.best_params_)

In [ ]:
# Best CV score

print("Best Cross-Validation Accuracy:",
      grid_nb.best_score_)

In [ ]:
# Best model

best_nb = grid_nb.best_estimator_

# Predictions
y_pred_best_nb = best_nb.predict(X_test)

In [ ]:
# Accuracy

print("Tuned Naive Bayes Accuracy:",
      accuracy_score(y_test, y_pred_best_nb))

In [ ]:
# Classification report

print(classification_report(y_test, y_pred_best_nb))

# Hyperparameter Tuning — Multinomial Naive Bayes

Hyperparameter tuning was performed to optimize the performance of the Multinomial Naive Bayes classifier.

The tuning process focused on identifying the optimal smoothing parameter (`alpha`) that would improve the model’s ability to generalize on unseen news articles.

---

# Best Hyperparameter

The optimal smoothing value identified through cross-validation was:

- Alpha = 0.1

---

# Best Cross-Validation Accuracy

The tuned model achieved a cross-validation accuracy of:

- 97.41%

This indicates excellent model stability and strong generalization capability across validation folds.

---

# Tuned Model Accuracy

After hyperparameter optimization, the tuned Naive Bayes model achieved:

- Test Accuracy = 97.65%

This represents a noticeable improvement over the baseline model performance.

---

# Classification Report

| Category | Precision | Recall | F1-Score |
|---|---|---|---|
| Business | 0.95 | 0.97 | 0.96 |
| Entertainment | 0.99 | 0.99 | 0.99 |
| Politics | 0.99 | 0.98 | 0.98 |
| Sports | 1.00 | 1.00 | 1.00 |
| Technology | 0.96 | 0.94 | 0.95 |

---

# Interpretation of Results

## 1. Strong Improvement After Tuning

The tuned model improved the overall classification accuracy compared to the baseline Naive Bayes model.

This demonstrates that hyperparameter optimization successfully enhanced the model’s predictive performance and generalization ability.

---

## 2. Outstanding Performance Across Categories

The model achieved very high precision, recall, and F1-scores across all news categories.

In particular:
- Sports achieved perfect classification performance.
- Entertainment and Politics achieved near-perfect performance.
- Business and Technology also showed highly reliable classification accuracy.

---

## 3. Sports Category Achieved Perfect Classification

The Sports category achieved:
- Precision = 1.00
- Recall = 1.00
- F1-score = 1.00

This indicates that sports-related articles contain highly distinctive vocabulary patterns that are easily identifiable by the model.

---

## 4. Minor Overlap in Technology Category

The Technology category showed slightly lower recall compared to other categories.

This may occur because technology articles sometimes overlap with:
- business-related topics
- entertainment and gaming content
- digital media discussions

Despite this, overall performance remains exceptionally strong.

---

# NLP Perspective

The strong performance of the tuned model confirms that:
- TF-IDF features effectively captured contextual information
- preprocessing successfully reduced textual noise
- category-specific vocabulary patterns were highly informative

The tuned Naive Bayes classifier demonstrated excellent capability in handling:
- sparse textual data
- high-dimensional feature spaces
- multi-class NLP classification problems

---

# Business Perspective

The highly accurate classification model can help FlipItNews:
- automate article categorization
- improve content organization
- enhance personalized recommendations
- optimize news feed management
- reduce manual tagging efforts

Accurate automated classification improves both operational efficiency and user experience.

---

# Overall Conclusion

The tuned Multinomial Naive Bayes model achieved outstanding classification performance after hyperparameter optimization.

Key achievements include:
- excellent generalization performance
- highly accurate category prediction
- strong performance across all news domains

The model establishes a strong baseline for further comparison with advanced NLP classifiers such as:
- Logistic Regression
- Linear SVM

---

## Model-2 - Logistic Regression: NLP Model Building using TF-IDF Features

In [ ]:
# =========================
# Logistic Regression Model
# =========================

from sklearn.linear_model import LogisticRegression

# Initialize model
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train model
lr_model.fit(X_train, y_train)

# Predictions
y_pred_lr = lr_model.predict(X_test)

In [ ]:
# Accuracy

lr_accuracy = accuracy_score(y_test, y_pred_lr)

print("Logistic Regression Accuracy:", lr_accuracy)

In [ ]:
# Classification report

print(classification_report(y_test, y_pred_lr))

In [ ]:
# Confusion matrix

cm_lr = confusion_matrix(y_test, y_pred_lr)

cm_lr

In [ ]:
# Heatmap

plt.figure(figsize=(8,6))

sns.heatmap(
    cm_lr,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=lr_model.classes_,
    yticklabels=lr_model.classes_
)

plt.xlabel('Predicted')

plt.ylabel('Actual')

plt.title('Logistic Regression Confusion Matrix')

plt.show()

# Logistic Regression Model Evaluation

A Logistic Regression classifier was trained using TF-IDF vectorized textual features to classify news articles into their respective categories.

Logistic Regression is widely used in NLP applications because of its:
- strong performance on high-dimensional sparse data
- computational efficiency
- ability to create stable linear decision boundaries

---

# Model Accuracy

The Logistic Regression model achieved:

- Accuracy = 97.42%

This indicates excellent classification performance on unseen news articles.

---

# Classification Report

| Category | Precision | Recall | F1-Score |
|---|---|---|---|
| Business | 0.92 | 0.98 | 0.95 |
| Entertainment | 0.99 | 0.97 | 0.98 |
| Politics | 1.00 | 0.95 | 0.97 |
| Sports | 1.00 | 1.00 | 1.00 |
| Technology | 0.99 | 0.96 | 0.97 |

---

# Interpretation of Results

## 1. Excellent Overall Performance

The Logistic Regression model achieved very high performance across all categories with:
- strong precision
- high recall
- excellent F1-scores

This demonstrates that:
- TF-IDF features effectively captured contextual information
- the preprocessing pipeline successfully reduced textual noise
- category-specific vocabulary patterns are highly distinguishable

---

## 2. Sports Category Achieved Perfect Classification

The Sports category achieved:
- Precision = 1.00
- Recall = 1.00
- F1-score = 1.00

This indicates that sports articles contain highly distinctive textual patterns and vocabulary that are easily separable from other categories.

---

## 3. Strong Performance in Entertainment and Technology

Both Entertainment and Technology categories achieved near-perfect precision and recall values.

This suggests that:
- domain-specific terminology is highly informative
- TF-IDF successfully emphasized category-relevant words
- the model effectively learned contextual patterns within these categories

---

## 4. Minor Misclassification in Business and Politics

A few Business and Politics articles were misclassified, likely due to overlapping vocabulary involving:
- government policies
- economic discussions
- financial reporting
- public affairs

However, the overall performance remains highly robust.

---

# Confusion Matrix Insights

The confusion matrix demonstrates that:
- most articles were correctly classified
- misclassification rates are very low
- category separation is highly effective

Notable observations include:
- Sports articles were classified perfectly
- Entertainment and Technology showed minimal confusion
- Business and Politics exhibited slight overlap

Overall, the model demonstrated strong category discrimination capability.

---

# NLP Perspective

The strong performance of Logistic Regression confirms that:
- TF-IDF provides highly informative textual features
- sparse high-dimensional representations work effectively with linear classifiers
- preprocessing and feature engineering significantly improved model quality

Logistic Regression performed exceptionally well in handling:
- sparse textual data
- multi-class classification
- high-dimensional NLP feature spaces

---

# Business Perspective

The highly accurate classification model can help FlipItNews:
- automate article categorization
- improve personalized news recommendations
- enhance content organization
- optimize search relevance
- improve user experience through faster content tagging

Such automated systems reduce manual effort and improve scalability for large news platforms.

---

#Overall Conclusion

The Logistic Regression model achieved outstanding classification performance using TF-IDF textual features.

Key achievements include:
- high classification accuracy
- strong generalization ability
- minimal category confusion
- robust performance across all news domains

The model establishes itself as a highly effective NLP classifier for automated news categorization tasks.

---

In [ ]:
# =========================
# Hyperparameter Tuning
# Logistic Regression
# =========================

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

# Parameter grid
param_grid_lr = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear']
}

# GridSearchCV
grid_lr = GridSearchCV(
    LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    param_grid_lr,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

# Fit grid search
grid_lr.fit(X_train, y_train)

In [ ]:
# Best parameters

print("Best Parameters:", grid_lr.best_params_)

In [ ]:
# Best CV score

print("Best Cross-Validation Accuracy:",
      grid_lr.best_score_)

In [ ]:
# Best model

best_lr = grid_lr.best_estimator_

# Predictions
y_pred_best_lr = best_lr.predict(X_test)

In [ ]:
# Accuracy

print("Tuned Logistic Regression Accuracy:",
      accuracy_score(y_test, y_pred_best_lr))

In [ ]:
# Classification report

print(classification_report(y_test, y_pred_best_lr))

In [ ]:
# Confusion matrix

cm_best_lr = confusion_matrix(
    y_test,
    y_pred_best_lr
)

cm_best_lr

In [ ]:
# Heatmap

plt.figure(figsize=(8,6))

sns.heatmap(
    cm_best_lr,
    annot=True,
    fmt='d',
    cmap='Reds',
    xticklabels=best_lr.classes_,
    yticklabels=best_lr.classes_
)

plt.xlabel('Predicted')

plt.ylabel('Actual')

plt.title('Tuned Logistic Regression Confusion Matrix')

plt.show()

# Hyperparameter Tuning — Logistic Regression

Hyperparameter tuning was performed on the Logistic Regression classifier to optimize model performance and improve generalization capability on unseen news articles.

The tuning process focused on identifying the best combination of:
- regularization strength (`C`)
- optimization solver

---

# Best Hyperparameters

The optimal parameters identified through GridSearchCV were:

- C = 10
- Solver = liblinear

These parameters provided the best balance between:
- model flexibility
- regularization
- classification performance

---

# Best Cross-Validation Accuracy

The tuned model achieved a cross-validation accuracy of:

- 97.24%

This indicates strong model stability and consistent performance across validation folds.

---

# Tuned Model Accuracy

After hyperparameter optimization, the tuned Logistic Regression model achieved:

- Test Accuracy = 97.42%

The tuned model maintained excellent classification performance across all news categories.

---

# Classification Report

| Category | Precision | Recall | F1-Score |
|---|---|---|---|
| Business | 0.93 | 0.97 | 0.95 |
| Entertainment | 0.99 | 0.96 | 0.97 |
| Politics | 0.99 | 0.98 | 0.98 |
| Sports | 0.99 | 1.00 | 1.00 |
| Technology | 0.99 | 0.96 | 0.97 |

---

# Interpretation of Results

## 1. Excellent Overall Classification Performance

The tuned Logistic Regression model achieved very high accuracy and strong category-level performance.

This demonstrates that:
- TF-IDF features effectively captured contextual patterns
- preprocessing successfully reduced textual noise
- the model generalized well on unseen news articles

---

## 2. Sports Category Achieved Perfect Recall

The Sports category achieved:
- Recall = 1.00
- F1-score = 1.00

This indicates that sports-related articles contain highly distinctive vocabulary patterns that are easily identifiable by the classifier.

---

## 3. Strong Performance Across All Categories

Politics, Entertainment, and Technology categories achieved near-perfect precision and recall values.

This suggests that:
- category-specific vocabulary was highly informative
- TF-IDF weighting effectively emphasized important terms
- Logistic Regression successfully separated overlapping classes

---

## 4. Minimal Category Confusion

The confusion matrix demonstrates that:
- the majority of articles were correctly classified
- misclassification rates remained very low
- category separation was highly effective

Minor overlap was observed primarily between:
- Business and Politics
- Technology and Business

which is expected because these domains occasionally share contextual vocabulary.

---

# NLP Perspective

The strong performance of the tuned Logistic Regression model confirms that:
- linear classifiers perform exceptionally well on sparse TF-IDF feature spaces
- preprocessing and feature engineering significantly improved model quality
- TF-IDF effectively captured discriminative vocabulary patterns

The model demonstrated excellent capability in handling:
- sparse matrices
- high-dimensional text data
- multi-class NLP classification problems

---

# Business Perspective

The highly accurate tuned Logistic Regression classifier can help FlipItNews:
- automate news categorization
- improve recommendation systems
- optimize content organization
- enhance user personalization
- reduce manual content tagging effort

Accurate classification improves both operational efficiency and user experience.

---

# Overall Conclusion

The tuned Logistic Regression model achieved excellent classification performance using TF-IDF textual features.

Key achievements include:
- strong generalization capability
- minimal category confusion
- high classification accuracy
- robust category-level performance

The model further confirms that:
# TF-IDF combined with linear classifiers

is highly effective for automated NLP-based news classification systems.

---

## Model-3 - Liner SVM: NLP Model Building using TF-IDF Features

In [ ]:
# =========================
# Linear SVM Model
# =========================

from sklearn.svm import LinearSVC

# Initialize model
svm_model = LinearSVC(random_state=42)

# Train model
svm_model.fit(X_train, y_train)

# Predictions
y_pred_svm = svm_model.predict(X_test)

In [ ]:
# Accuracy

svm_accuracy = accuracy_score(y_test, y_pred_svm)

print("Linear SVM Accuracy:", svm_accuracy)

In [ ]:
# Classification report

print(classification_report(y_test, y_pred_svm))

In [ ]:
# Confusion matrix

cm_svm = confusion_matrix(y_test, y_pred_svm)

cm_svm

In [ ]:
# Heatmap

plt.figure(figsize=(8,6))

sns.heatmap(
    cm_svm,
    annot=True,
    fmt='d',
    cmap='Purples',
    xticklabels=svm_model.classes_,
    yticklabels=svm_model.classes_
)

plt.xlabel('Predicted')

plt.ylabel('Actual')

plt.title('Linear SVM Confusion Matrix')

plt.show()

# Linear SVM Model Evaluation

A Linear Support Vector Machine (Linear SVM) classifier was trained using TF-IDF vectorized textual features to classify news articles into their respective categories.

Linear SVM is one of the most effective classical machine learning algorithms for NLP tasks because of its ability to:
- handle sparse high-dimensional data efficiently
- create strong linear decision boundaries
- reduce category overlap
- generalize well on textual datasets

---

# Model Accuracy

The Linear SVM model achieved:

- Accuracy = 97.65%

This indicates outstanding classification performance on unseen news articles.

---

# Classification Report

| Category | Precision | Recall | F1-Score |
|---|---|---|---|
| Business | 0.94 | 0.97 | 0.96 |
| Entertainment | 0.99 | 0.97 | 0.98 |
| Politics | 0.99 | 0.98 | 0.98 |
| Sports | 0.99 | 1.00 | 1.00 |
| Technology | 0.99 | 0.96 | 0.97 |

---

# Interpretation of Results

## 1. Excellent Overall Performance

The Linear SVM classifier achieved extremely high performance across all categories with:
- strong precision
- high recall
- excellent F1-scores

This demonstrates that:
- TF-IDF features effectively captured semantic patterns
- preprocessing successfully reduced textual noise
- category-specific vocabulary was highly informative

---

## 2. Sports Category Achieved Perfect Recall

The Sports category achieved:
- Recall = 1.00
- F1-score = 1.00

This indicates that sports articles contain highly distinctive textual patterns that are easily identifiable by the classifier.

The consistent vocabulary related to:
- games
- players
- tournaments
- wins

likely contributed significantly to the model’s performance.

---

## 3. Strong Performance Across All Categories

Entertainment, Politics, and Technology categories achieved near-perfect precision and recall values.

This suggests that:
- category-specific terminology is highly discriminative
- TF-IDF weighting effectively emphasized important contextual words
- Linear SVM successfully separated overlapping categories

---

## 4. Minor Misclassification in Business and Technology

A few Business and Technology articles were misclassified due to potential overlap involving:
- economic technology discussions
- digital business topics
- media and technology reporting

However, the overall misclassification rate remains extremely low.

---

# Confusion Matrix Insights

The confusion matrix shows that:
- the majority of articles were correctly classified
- category separation is highly effective
- misclassifications are minimal

Key observations include:
- Sports articles were classified almost perfectly
- Politics and Entertainment demonstrated highly stable classification
- Technology showed very small overlap with Business

Overall, the classifier demonstrated excellent discrimination capability.

---

# NLP Perspective

The strong performance of Linear SVM confirms that:
- TF-IDF vectorization is highly effective for NLP classification tasks
- sparse high-dimensional text representations work exceptionally well with linear classifiers
- preprocessing significantly improved feature quality

Linear SVM proved highly effective in handling:
- sparse matrices
- large vocabulary spaces
- multi-class text classification problems

---

# Business Perspective

The highly accurate Linear SVM model can significantly benefit FlipItNews by enabling:
- automated news categorization
- personalized article recommendations
- improved content organization
- efficient topic classification
- scalable content management

Accurate automated classification enhances user experience while reducing manual effort and operational complexity.

---

# Overall Conclusion

The Linear SVM model achieved outstanding classification performance using TF-IDF textual features.

Key achievements include:
- excellent accuracy
- strong category separation
- minimal misclassification
- robust generalization capability

The model demonstrates that Linear SVM is highly suitable for large-scale NLP-based news classification systems and establishes itself as one of the strongest-performing models in this study.

---

## Hyperparameter Tuning of Linear SVM Using GridSearchCV

In [ ]:
# =========================
# Hyperparameter Tuning
# Linear SVM
# =========================

from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC

# Parameter grid
param_grid_svm = {
    'C': [0.01, 0.1, 1, 10]
}

# GridSearchCV
grid_svm = GridSearchCV(
    LinearSVC(random_state=42),
    param_grid_svm,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

# Fit grid search
grid_svm.fit(X_train, y_train)

In [ ]:
# Best parameters

print("Best Parameters:", grid_svm.best_params_)

In [ ]:
# Best CV score

print("Best Cross-Validation Accuracy:",
      grid_svm.best_score_)

In [ ]:
# Best model

best_svm = grid_svm.best_estimator_

# Predictions
y_pred_best_svm = best_svm.predict(X_test)

In [ ]:
# Accuracy

print("Tuned Linear SVM Accuracy:",
      accuracy_score(y_test, y_pred_best_svm))

In [ ]:
# Classification report

print(classification_report(y_test, y_pred_best_svm))

In [ ]:
# Confusion Matrix

cm_best_svm = confusion_matrix(
    y_test,
    y_pred_best_svm
)

cm_best_svm

In [ ]:
# Heatmap

plt.figure(figsize=(8,6))

sns.heatmap(
    cm_best_svm,
    annot=True,
    fmt='d',
    cmap='Greys',
    xticklabels=best_svm.classes_,
    yticklabels=best_svm.classes_
)

plt.xlabel('Predicted')

plt.ylabel('Actual')

plt.title('Tuned Linear SVM Confusion Matrix')

plt.show()

# Hyperparameter Tuning — Linear SVM

Hyperparameter tuning was performed on the Linear SVM classifier to optimize classification performance and improve generalization capability on unseen news articles.

The tuning process focused on identifying the optimal value of the regularization parameter (`C`) using GridSearchCV.

---

# Best Hyperparameter

The optimal parameter identified through cross-validation was:

- C = 10

This value provided the best balance between:
- model flexibility
- classification accuracy
- regularization strength
- decision boundary optimization

---

# Best Cross-Validation Accuracy

The tuned model achieved a cross-validation accuracy of:

- 97.47%

This indicates excellent stability and strong consistency across validation folds.

---

# Tuned Model Accuracy

After hyperparameter optimization, the tuned Linear SVM model achieved:

- Test Accuracy = 97.89%

This represents the highest overall performance among all evaluated models in this NLP business case study.

---

# Classification Report

| Category | Precision | Recall | F1-Score |
|---|---|---|---|
| Business | 0.95 | 0.97 | 0.96 |
| Entertainment | 0.99 | 0.97 | 0.98 |
| Politics | 0.99 | 0.99 | 0.99 |
| Sports | 0.99 | 1.00 | 1.00 |
| Technology | 0.99 | 0.96 | 0.97 |

---

# Confusion Matrix

| Actual \ Predicted | Business | Entertainment | Politics | Sports | Technology |
|---|---|---|---|---|---|
| Business | 98 | 1 | 1 | 0 | 1 |
| Entertainment | 1 | 72 | 0 | 1 | 0 |
| Politics | 1 | 0 | 80 | 0 | 0 |
| Sports | 0 | 0 | 0 | 101 | 0 |
| Technology | 3 | 0 | 0 | 0 | 66 |

---

# Interpretation of Results

## 1. Best Overall Model Performance

The tuned Linear SVM achieved the highest overall accuracy among all tested machine learning models.

This demonstrates that:
- Linear SVM is highly effective for sparse TF-IDF textual data
- preprocessing successfully captured meaningful linguistic patterns
- category-specific vocabulary is highly distinguishable

---

## 2. Excellent Category-Level Classification

All categories achieved exceptionally high:
- precision
- recall
- F1-scores

This indicates:
- strong model stability
- highly accurate category separation
- minimal classification confusion

---

## 3. Sports Category Achieved Perfect Classification

The Sports category achieved:
- Recall = 1.00
- F1-score = 1.00

Additionally:
- all 101 Sports articles were correctly classified

This confirms that sports-related articles contain highly distinctive textual patterns and domain-specific vocabulary.

---

## 4. Politics Category Achieved Near-Perfect Performance

Politics articles achieved:
- Precision = 0.99
- Recall = 0.99

Only one Politics article was misclassified, demonstrating extremely strong semantic separation for political news content.

---

## 5. Minimal Misclassification Across Categories

The confusion matrix shows that:
- the majority of articles were classified correctly
- category overlap was extremely low
- misclassification counts were minimal

Minor overlaps were observed between:
- Business and Technology
- Business and Politics

This is expected because these domains occasionally share:
- economic terminology
- policy discussions
- market-related content

---

## 6. Technology Category Performance

Technology articles achieved:
- Precision = 0.99
- Recall = 0.96

A few Technology articles were classified as Business articles, likely because:
- technology news often includes corporate and financial reporting
- business and technology domains share contextual vocabulary

Despite this, overall performance remained exceptionally strong.

---

# NLP Perspective

The superior performance of the tuned Linear SVM confirms that:
- TF-IDF vectorization is highly effective for NLP classification
- sparse high-dimensional textual representations work exceptionally well with linear classifiers
- hyperparameter tuning significantly improves classification boundaries and model generalization

The model demonstrated excellent capability in handling:
- sparse matrices
- large vocabulary spaces
- multi-class text classification problems

---

# Business Perspective

The highly accurate tuned Linear SVM model can significantly benefit FlipItNews by enabling:
- automated article categorization
- personalized news recommendations
- improved content organization
- efficient topic classification
- scalable news management systems

Such automation improves:
- operational efficiency
- recommendation quality
- user experience
- content discovery

while reducing manual classification effort.

---

# Overall Conclusion

The tuned Linear SVM model achieved the best overall classification performance in the FlipItNews NLP business case study.

Key achievements include:
- highest overall accuracy
- minimal category confusion
- excellent generalization capability
- highly stable multi-class classification performance

The results strongly confirm that:
# TF-IDF + Linear SVM

is the most effective model combination for automated news classification within this dataset.

---
---

## Model-4 - Random Forest: NLP Model Building using TF-IDF Features

In [ ]:
# =========================
# Random Forest Classifier
# =========================

from sklearn.ensemble import RandomForestClassifier

# Initialize model
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train model
rf_model.fit(X_train, y_train)

# Predictions
y_pred_rf = rf_model.predict(X_test)

In [ ]:
# Accuracy

rf_accuracy = accuracy_score(y_test, y_pred_rf)

print("Random Forest Accuracy:", rf_accuracy)

In [ ]:
# Classification report

print(classification_report(y_test, y_pred_rf))

In [ ]:
# Confusion matrix

cm_rf = confusion_matrix(y_test, y_pred_rf)

cm_rf

In [ ]:
# Heatmap

plt.figure(figsize=(8,6))

sns.heatmap(
    cm_rf,
    annot=True,
    fmt='d',
    cmap='Oranges',
    xticklabels=rf_model.classes_,
    yticklabels=rf_model.classes_
)

plt.xlabel('Predicted')

plt.ylabel('Actual')

plt.title('Random Forest Confusion Matrix')

plt.show()

# Random Forest Model Evaluation

A Random Forest classifier was trained using TF-IDF vectorized textual features to classify news articles into their respective categories.

Random Forest is an ensemble learning algorithm that combines multiple decision trees to improve prediction stability and reduce overfitting.

Although tree-based algorithms are generally more effective for structured tabular datasets, Random Forest was evaluated to compare its performance against other NLP classifiers such as:
- Naive Bayes
- Logistic Regression
- Linear SVM

---

# Model Accuracy

The Random Forest model achieved:

- Accuracy = 95.54%

This indicates strong overall classification performance on unseen news articles.

---

# Classification Report

| Category | Precision | Recall | F1-Score |
|---|---|---|---|
| Business | 0.90 | 0.97 | 0.93 |
| Entertainment | 1.00 | 0.95 | 0.97 |
| Politics | 0.99 | 0.94 | 0.96 |
| Sports | 0.95 | 1.00 | 0.98 |
| Technology | 0.97 | 0.90 | 0.93 |

---

# Interpretation of Results

## 1. Strong Overall Performance

The Random Forest classifier achieved high classification accuracy with strong precision, recall, and F1-scores across all categories.

This indicates that:
- TF-IDF features successfully captured meaningful textual patterns
- the preprocessing pipeline effectively reduced textual noise
- the model was able to learn category-specific vocabulary relationships

---

## 2. Sports Category Achieved Perfect Recall

The Sports category achieved:
- Recall = 1.00
- F1-score = 0.98

This suggests that sports-related articles contain highly distinctive vocabulary and contextual patterns that are easy for the classifier to identify.

---

## 3. Entertainment and Politics Performed Very Well

Both Entertainment and Politics categories achieved strong performance due to:
- domain-specific terminology
- strong contextual separation
- well-defined reporting styles

The presence of category-specific words likely improved classification quality.

---

## 4. Slightly Lower Performance in Business and Technology

Business and Technology categories showed comparatively lower precision and recall values.

This may occur because:
- technology and business topics sometimes overlap
- economic and digital reporting share similar vocabulary
- Random Forest struggles slightly with sparse high-dimensional NLP data

Despite this, the performance remains highly competitive.

---

# Confusion Matrix Insights

The confusion matrix demonstrates that:
- most articles were correctly classified
- category separation remained strong
- misclassification rates were relatively low

Key observations include:
- Sports articles were classified perfectly
- Technology articles occasionally overlapped with Business and Sports
- Politics articles showed minor overlap with Business

Overall, the model demonstrated strong multi-class classification capability.

---

# NLP Perspective

The Random Forest model performed well despite the challenges associated with:
- sparse textual data
- high-dimensional TF-IDF feature spaces

However, compared to:
- Logistic Regression
- Linear SVM

the model showed slightly weaker performance, which is expected in NLP tasks involving sparse matrices.

This highlights that:
- linear models often perform better on TF-IDF features
- tree-based models are less efficient for very high-dimensional text data

---

# Business Perspective

The Random Forest classifier can still provide valuable business applications such as:
- automated news categorization
- recommendation systems
- content organization
- topic classification

Its ensemble structure also offers:
- stable predictions
- reduced overfitting
- strong generalization capability

---

# Overall Conclusion

The Random Forest model achieved strong classification performance using TF-IDF textual features.

Key achievements include:
- high overall accuracy
- strong category-level performance
- effective handling of multi-class classification

However, compared to:
- Logistic Regression
- Linear SVM

the Random Forest classifier demonstrated slightly lower effectiveness for sparse NLP feature spaces.

This confirms that linear models remain more suitable for TF-IDF-based news classification tasks.

---

## Hyperparameter Tuning for Random Forest Using GridSearchCV

In [ ]:
# =========================
# Hyperparameter Tuning
# Random Forest
# =========================

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Parameter grid
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

# GridSearchCV
grid_rf = GridSearchCV(
    RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ),
    param_grid_rf,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

# Fit grid search
grid_rf.fit(X_train, y_train)

In [ ]:
# Best parameters

print("Best Parameters:", grid_rf.best_params_)

In [ ]:
# Best CV score

print("Best Cross-Validation Accuracy:",
      grid_rf.best_score_)

In [ ]:
# Best model

best_rf = grid_rf.best_estimator_

# Predictions
y_pred_best_rf = best_rf.predict(X_test)

In [ ]:
# Accuracy

print("Tuned Random Forest Accuracy:",
      accuracy_score(y_test, y_pred_best_rf))

In [ ]:
# Classification report

print(classification_report(y_test, y_pred_best_rf))

In [ ]:
# Confusion matrix

cm_best_rf = confusion_matrix(
    y_test,
    y_pred_best_rf
)

cm_best_rf

In [ ]:
# Heatmap

plt.figure(figsize=(8,6))

sns.heatmap(
    cm_best_rf,
    annot=True,
    fmt='d',
    cmap='Oranges',
    xticklabels=best_rf.classes_,
    yticklabels=best_rf.classes_
)

plt.xlabel('Predicted')

plt.ylabel('Actual')

plt.title('Tuned Random Forest Confusion Matrix')

plt.show()

# Hyperparameter Tuning — Random Forest

Hyperparameter tuning was performed on the Random Forest classifier to optimize model performance and improve generalization capability on unseen news articles.

The tuning process focused on optimizing:
- number of trees (`n_estimators`)
- tree depth (`max_depth`)
- minimum samples required for splitting (`min_samples_split`)

using GridSearchCV.

---

# Best Hyperparameters

The optimal parameters identified through GridSearchCV were:

- Number of Trees (`n_estimators`) = 100
- Maximum Depth (`max_depth`) = None
- Minimum Samples Split (`min_samples_split`) = 5

These parameters provided the best balance between:
- model complexity
- classification performance
- generalization capability

---

# Best Cross-Validation Accuracy

The tuned model achieved a cross-validation accuracy of:

- 96.12%

This indicates strong and stable performance across validation folds.

---

# Tuned Model Accuracy

After hyperparameter optimization, the tuned Random Forest model achieved:

- Test Accuracy = 95.77%

This represents a slight improvement over the baseline Random Forest model.

---

# Classification Report

| Category | Precision | Recall | F1-Score |
|---|---|---|---|
| Business | 0.91 | 0.98 | 0.94 |
| Entertainment | 1.00 | 0.93 | 0.97 |
| Politics | 0.99 | 0.95 | 0.97 |
| Sports | 0.96 | 1.00 | 0.98 |
| Technology | 0.95 | 0.90 | 0.93 |

---

# Confusion Matrix

| Actual \ Predicted | Business | Entertainment | Politics | Sports | Technology |
|---|---|---|---|---|---|
| Business | 99 | 0 | 0 | 0 | 2 |
| Entertainment | 3 | 69 | 1 | 0 | 1 |
| Politics | 3 | 0 | 77 | 1 | 0 |
| Sports | 0 | 0 | 0 | 101 | 0 |
| Technology | 4 | 0 | 0 | 3 | 62 |

---

# Interpretation of Results

## 1. Strong Overall Classification Performance

The tuned Random Forest model achieved strong classification accuracy with high precision, recall, and F1-scores across all categories.

This demonstrates that:
- TF-IDF features effectively captured important textual patterns
- preprocessing successfully reduced textual noise
- ensemble learning improved classification robustness

---

## 2. Sports Category Achieved Perfect Recall

The Sports category achieved:
- Recall = 1.00
- F1-score = 0.98

All Sports articles were correctly classified, indicating highly distinctive sports-related vocabulary patterns.

---

## 3. Strong Performance in Business and Politics

Business and Politics categories achieved strong classification performance with minimal confusion.

The model successfully captured:
- financial terminology
- political reporting patterns
- government and economic vocabulary

---

## 4. Technology Category Showed Slight Overlap

Technology articles achieved:
- Precision = 0.95
- Recall = 0.90

Some Technology articles were classified as:
- Business
- Sports

This may occur because:
- technology reporting often overlaps with business news
- gaming and sports-related technology articles share contextual vocabulary

---

## 5. Minimal Overall Misclassification

The confusion matrix demonstrates that:
- the majority of articles were classified correctly
- category overlap remained relatively low
- ensemble learning handled multi-class classification effectively

However, compared to:
- Logistic Regression
- Linear SVM

Random Forest still showed slightly higher category confusion.

---

# NLP Perspective

The tuned Random Forest classifier performed well despite the challenges associated with:
- sparse TF-IDF matrices
- high-dimensional textual feature spaces

However, the results confirm that:
- linear classifiers are generally more effective for sparse NLP data
- tree-based models are comparatively less efficient for very large vocabulary spaces

This explains why:
- Linear SVM
- Logistic Regression

outperformed Random Forest in this study.

---

# Business Perspective

The tuned Random Forest model can still provide valuable business applications such as:
- automated article categorization
- topic classification
- recommendation systems
- scalable content organization

The ensemble structure also provides:
- stable predictions
- reduced overfitting
- strong generalization capability

---

# Overall Conclusion

The tuned Random Forest model achieved strong classification performance using TF-IDF textual features.

Key achievements include:
- strong overall accuracy
- effective multi-class classification
- stable ensemble learning performance

However, compared to:
- Linear SVM
- Logistic Regression

the Random Forest classifier demonstrated slightly weaker performance on sparse high-dimensional NLP data.

The results further confirm that:
# TF-IDF combined with linear classifiers

is more effective for automated news classification tasks in the FlipItNews dataset.

---

# Final Model Performance Comparison

Multiple machine learning models were trained and optimized using TF-IDF textual features to classify news articles into their respective categories.

Hyperparameter tuning was performed using:
# GridSearchCV

to improve:
- model generalization
- classification accuracy
- category-level performance

The final tuned models evaluated in this study were:
- Multinomial Naive Bayes
- Logistic Regression
- Linear SVM
- Random Forest

---

# Final Tuned Model Comparison

| Model | Best Parameters | Cross-Validation Accuracy | Test Accuracy |
|---|---|---|---|
| Tuned Naive Bayes | Alpha = 0.1 | 97.41% | 97.65% |
| Tuned Logistic Regression | C = 10 | 97.24% | 97.42% |
| Tuned Linear SVM | C = 10 | 97.47% | 97.89% |
| Tuned Random Forest | n_estimators = 100, max_depth = None, min_samples_split = 5 | 96.12% | 95.77% |

---

# Interpretation of Overall Results

## 1. Linear SVM Achieved the Best Overall Performance

The tuned Linear SVM achieved:
- Highest test accuracy
- Strongest overall generalization
- Minimal category confusion

This confirms that:
# Linear SVM is highly effective for sparse TF-IDF textual data.

The model successfully handled:
- high-dimensional feature spaces
- sparse matrices
- overlapping textual categories

with exceptional stability and accuracy.

---

# Best Performing Model

| Model | Accuracy |
|---|---|
| Tuned Linear SVM | 97.89% |

The tuned Linear SVM emerged as the:
# Best-performing classifier

for the FlipItNews news classification problem.

---

# 2. Naive Bayes Performed Exceptionally Well

The tuned Multinomial Naive Bayes model achieved:
- 97.65% accuracy

This demonstrates that probabilistic classifiers remain highly effective for:
- TF-IDF-based NLP tasks
- sparse textual datasets
- multi-class text classification

Naive Bayes also provided:
- fast training
- computational efficiency
- strong baseline performance

---

# 3. Logistic Regression Delivered Strong and Stable Results

The tuned Logistic Regression model achieved:
- 97.42% accuracy

The model demonstrated:
- excellent generalization capability
- highly stable classification boundaries
- strong precision and recall across all categories

Logistic Regression proved highly effective in handling:
- sparse textual features
- category-level semantic separation

---

# 4. Random Forest Performed Well but Slightly Lower

The tuned Random Forest model achieved:
- 95.77% accuracy

Although ensemble learning improved prediction stability, the model showed slightly weaker performance compared to linear classifiers.

This is expected because:
- Random Forest struggles with sparse high-dimensional TF-IDF data
- tree-based models are less efficient for very large vocabulary spaces

However, the model still demonstrated:
- strong multi-class classification capability
- stable predictions
- good generalization performance

---

# NLP Perspective

The overall results strongly confirm that:
# TF-IDF combined with linear classifiers

is highly effective for automated NLP-based news classification.

The study demonstrates that:
- sparse textual representations are best handled by linear models
- preprocessing significantly improves model performance
- TF-IDF successfully captures category-specific contextual information

The following techniques contributed significantly to model success:
- text cleaning
- tokenization
- stopword removal
- lemmatization
- TF-IDF vectorization

---

# Business Perspective

The highly accurate classification models can significantly benefit FlipItNews by enabling:
- automated article categorization
- personalized news recommendations
- scalable content management
- improved search relevance
- efficient content organization

Automated classification systems help:
- reduce manual tagging effort
- improve operational efficiency
- enhance user experience
- support intelligent recommendation systems

---

# Key Business Insights

| Insight | Observation |
|---|---|
| Best Model | Tuned Linear SVM |
| Most Stable Linear Model | Logistic Regression |
| Fastest Probabilistic Model | Naive Bayes |
| Strong Ensemble Model | Random Forest |
| Best NLP Architecture | TF-IDF + Linear Classifiers |

---

# Overall Conclusion

The FlipItNews NLP classification pipeline successfully demonstrated that:
- textual preprocessing significantly improves feature quality
- TF-IDF effectively captures semantic and contextual information
- hyperparameter tuning enhances model generalization
- linear classifiers outperform ensemble tree-based methods for sparse NLP datasets

Among all evaluated models:
# Tuned Linear SVM achieved the best overall classification performance

with:
- highest accuracy
- minimal category confusion
- strong generalization capability

The study confirms that:
# TF-IDF + Linear SVM

is the most effective model combination for automated news classification within the FlipItNews dataset.

---

# Advanced NLP Modeling using Transformers

In [ ]:
import torch
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

## DistilBERT
We are using DistilBERT instead of full BERT because:

faster training

lighter architecture

lower memory usage

excellent NLP performance

In [ ]:
# =========================
# Encode Target Labels
# =========================

label_encoder = LabelEncoder()

df['Label'] = label_encoder.fit_transform(df['Category'])

# Check mapping
label_mapping = dict(
    zip(
        label_encoder.classes_,
        label_encoder.transform(label_encoder.classes_)
    )
)

print(label_mapping)

In [ ]:
# =========================
# Train-Test Split
# =========================

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['Final_Text'],
    df['Label'],
    test_size=0.2,
    random_state=42,
    stratify=df['Label']
)

print(X_train_text.shape)
print(X_test_text.shape)

In [ ]:
# =========================
# Load DistilBERT Tokenizer
# =========================

tokenizer = DistilBertTokenizerFast.from_pretrained(
    'distilbert-base-uncased'
)

In [ ]:
# =========================
# Tokenize Text Data
# =========================

train_encodings = tokenizer(
    list(X_train_text),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    list(X_test_text),
    truncation=True,
    padding=True,
    max_length=256
)

In [ ]:
# =========================
# Create HuggingFace Dataset
# =========================

train_dataset = Dataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': list(y_train)
})

test_dataset = Dataset.from_dict({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask'],
    'labels': list(y_test)
})

print(train_dataset)
print(test_dataset)

In [ ]:
# =========================
# Load DistilBERT Model
# =========================

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=5
)

In [ ]:
# =========================
# Evaluation Metrics
# =========================

def compute_metrics(pred):

    labels = pred.label_ids

    preds = pred.predictions.argmax(-1)

    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc
    }

In [ ]:
# =========================
# Training Arguments
# =========================

training_args = TrainingArguments(

    output_dir='./results',

    num_train_epochs=2,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    warmup_steps=100,

    weight_decay=0.01,

    logging_dir='./logs',

    logging_steps=50,

    eval_strategy='epoch',

    save_strategy='epoch',

    load_best_model_at_end=True,

    report_to='none'
)

In [ ]:
# =========================
# Trainer
# =========================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

In [ ]:
# =========================
# Train Model
# =========================

trainer.train()

In [ ]:
# =========================
# Predictions
# =========================

predictions = trainer.predict(test_dataset)

y_pred_bert = np.argmax(
    predictions.predictions,
    axis=1
)

In [ ]:
# Accuracy

bert_accuracy = accuracy_score(
    y_test,
    y_pred_bert
)

print("DistilBERT Accuracy:", bert_accuracy)

In [ ]:
# Classification Report

print(
    classification_report(
        y_test,
        y_pred_bert,
        target_names=label_encoder.classes_
    )
)

In [ ]:
# Confusion Matrix

cm_bert = confusion_matrix(
    y_test,
    y_pred_bert
)

cm_bert

In [ ]:
# Heatmap

plt.figure(figsize=(8,6))

sns.heatmap(
    cm_bert,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)

plt.xlabel('Predicted')

plt.ylabel('Actual')

plt.title('DistilBERT Confusion Matrix')

plt.show()

# DistilBERT Model Evaluation

An advanced transformer-based NLP model, DistilBERT, was implemented for automated news article classification.

DistilBERT is a lightweight and efficient version of BERT (Bidirectional Encoder Representations from Transformers) that retains strong contextual language understanding while reducing computational complexity.

Unlike traditional NLP approaches such as:
- Bag of Words (BoW)
- TF-IDF

DistilBERT captures:
# contextual semantics
# sentence-level meaning
# word relationships

through transformer-based embeddings and transfer learning.

---

# Transfer Learning Approach

A pretrained DistilBERT model:
# `distilbert-base-uncased`

was fine-tuned on the FlipItNews dataset for multi-class news classification.

The model leveraged:
- pretrained language knowledge
- contextual embeddings
- transformer attention mechanisms

to improve classification performance.

---

# DistilBERT Accuracy

The DistilBERT model achieved:

- Accuracy = 97.89%

This represents the highest overall performance among all evaluated models and demonstrates the effectiveness of transformer-based NLP architectures.

---

# Classification Report

| Category | Precision | Recall | F1-Score |
|---|---|---|---|
| Business | 0.96 | 0.95 | 0.96 |
| Entertainment | 0.99 | 1.00 | 0.99 |
| Politics | 0.98 | 0.99 | 0.98 |
| Sports | 1.00 | 1.00 | 1.00 |
| Technology | 0.97 | 0.96 | 0.96 |

---

# Confusion Matrix

| Actual \ Predicted | Business | Entertainment | Politics | Sports | Technology |
|---|---|---|---|---|---|
| Business | 96 | 1 | 2 | 0 | 2 |
| Entertainment | 0 | 74 | 0 | 0 | 0 |
| Politics | 1 | 0 | 80 | 0 | 0 |
| Sports | 0 | 0 | 0 | 101 | 0 |
| Technology | 3 | 0 | 0 | 0 | 66 |

---

# Interpretation of Results

## 1. Outstanding Overall Performance

DistilBERT achieved extremely high classification accuracy with excellent precision, recall, and F1-scores across all categories.

This demonstrates that transformer-based models effectively capture:
- contextual meaning
- semantic relationships
- sentence-level information

better than traditional frequency-based NLP methods.

---

## 2. Sports Category Achieved Perfect Classification

The Sports category achieved:
- Precision = 1.00
- Recall = 1.00
- F1-score = 1.00

All Sports articles were correctly classified, indicating highly distinctive vocabulary and contextual patterns within sports-related content.

---

## 3. Entertainment Category Achieved Perfect Recall

Entertainment articles achieved:
- Recall = 1.00

All Entertainment articles were correctly identified by the model, demonstrating strong contextual understanding of entertainment-related language patterns.

---

## 4. Minimal Category Confusion

The confusion matrix demonstrates:
- extremely low misclassification rates
- strong category separation
- excellent contextual learning capability

Minor overlap was observed primarily between:
- Business and Technology
- Business and Politics

which is expected because these domains occasionally share:
- financial terminology
- corporate reporting language
- economic discussions

---

## 5. Strong Performance Across All Categories

All categories achieved:
- precision above 0.95
- recall above 0.95
- highly stable F1-scores

This confirms that DistilBERT generalized exceptionally well on unseen news articles.

---

# Advanced NLP Perspective

DistilBERT significantly differs from classical NLP models because it uses:
# transformer architecture
# attention mechanisms
# contextual embeddings

rather than simple frequency-based feature representations.

Unlike TF-IDF and BoW:
- transformers understand context
- word meaning changes dynamically based on surrounding words
- semantic relationships are preserved

This enables more intelligent and context-aware classification.

---

# Transfer Learning Perspective

The model successfully utilized:
# transfer learning

by adapting pretrained language knowledge to the FlipItNews classification task.

Advantages of transfer learning include:
- reduced training time
- improved language understanding
- better contextual representation
- strong performance on relatively small datasets

---

# Comparison with Classical NLP Models

| Model | Accuracy |
|---|---|
| Tuned Naive Bayes | 97.65% |
| Tuned Logistic Regression | 97.42% |
| Tuned Linear SVM | 97.89% |
| Tuned Random Forest | 95.77% |
| DistilBERT | 97.89% |

DistilBERT achieved performance comparable to the best-performing classical NLP model:
# Tuned Linear SVM

However, DistilBERT offers additional advantages through:
- contextual understanding
- semantic learning
- transfer learning capability

which make transformer-based architectures highly valuable for advanced NLP applications.

---

# Business Perspective

The DistilBERT classifier can significantly enhance FlipItNews through:
- intelligent automated news categorization
- personalized recommendation systems
- semantic content understanding
- scalable content management
- improved search relevance

Transformer-based models also provide:
- superior contextual understanding
- better adaptability to language variations
- improved handling of nuanced textual content

This can substantially improve:
- user engagement
- recommendation quality
- operational efficiency

---

# Overall Conclusion

The DistilBERT transformer model achieved outstanding classification performance on the FlipItNews dataset.

Key achievements include:
- highest overall accuracy
- minimal category confusion
- strong contextual understanding
- highly stable category-level performance

The results demonstrate that:
# transformer-based NLP models

are highly effective for automated news classification tasks.

The study further confirms that:
# transfer learning + contextual embeddings

provide major advantages in modern NLP systems compared to traditional frequency-based approaches.

---

# Final Model Comparison: Classical NLP vs DistilBERT

This project evaluated both:
- classical machine learning NLP models
- advanced transformer-based NLP models

for automated news article classification using the FlipItNews dataset.

The objective was to compare:
- traditional frequency-based NLP approaches
- modern contextual transformer architectures

to identify the most effective solution for multi-class news classification.

---

# Models Evaluated

## Classical NLP Models

The following traditional machine learning models were trained using:
- TF-IDF vectorization
- Bag of Words concepts
- handcrafted preprocessing pipeline

### Models:
- Multinomial Naive Bayes
- Logistic Regression
- Linear SVM
- Random Forest

---

## Advanced NLP Model

The advanced NLP approach used:
## DistilBERT Transformer Model

leveraging:
- contextual embeddings
- transformer attention mechanisms
- transfer learning

---

# Final Accuracy Comparison

| Model | NLP Type | Accuracy |
|---|---|---|
| Tuned Naive Bayes | Classical NLP | 97.65% |
| Tuned Logistic Regression | Classical NLP | 97.42% |
| Tuned Linear SVM | Classical NLP | 97.89% |
| Tuned Random Forest | Classical NLP | 95.77% |
| DistilBERT | Transformer NLP | 97.89% |

---

# Best Performing Models

| Model | Accuracy |
|---|---|
| Tuned Linear SVM | 97.89% |
| DistilBERT | 97.89% |

Both models achieved the highest overall classification accuracy in this project.

---

# Classical NLP Approach

## Workflow

The classical NLP pipeline involved:
- text cleaning
- tokenization
- stopword removal
- lemmatization
- TF-IDF vectorization
- machine learning classification

---

## Strengths of Classical NLP

### 1. Computational Efficiency

Classical models:
- train much faster
- require lower computational resources
- are lightweight and scalable

especially:
- Naive Bayes
- Logistic Regression
- Linear SVM

---

### 2. Excellent Performance on Structured Text

TF-IDF combined with linear classifiers achieved extremely strong performance because:
- news categories contain domain-specific vocabulary
- category-level word distributions were highly separable
- preprocessing reduced noise effectively

---

### 3. Easier Interpretability

Classical NLP models allow:
- feature importance analysis
- top predictive word extraction
- easier explainability

This improves:
- model transparency
- interpretability
- business understanding

---

## Limitations of Classical NLP

Classical approaches rely primarily on:
- word frequency statistics

They do not fully understand:
- sentence meaning
- semantic context
- contextual word relationships

Example:
- "bank account"
- "bank river"

TF-IDF treats the word “bank” similarly regardless of context.

---

# Transformer-Based NLP Approach

## DistilBERT Workflow

DistilBERT used:
- pretrained transformer embeddings
- contextual token representations
- transfer learning
- attention mechanisms

Unlike classical NLP:
- no handcrafted feature engineering was required.

---

## Strengths of DistilBERT

### 1. Contextual Understanding

DistilBERT understands:
- sentence semantics
- contextual meaning
- relationships between words

This enables:
- smarter classification
- deeper language understanding
- improved handling of nuanced text

---

### 2. Transfer Learning Advantage

The pretrained DistilBERT model already possessed:
- general English language understanding
- semantic knowledge
- contextual representations

Fine-tuning adapted this knowledge specifically for:
- news classification

---

### 3. Minimal Preprocessing Requirement

Transformers naturally handle:
- word variations
- contextual semantics
- sentence structure

This reduces dependency on:
- heavy preprocessing
- manual feature engineering

---

## Limitations of DistilBERT

Despite strong performance, transformer models have certain tradeoffs:

### 1. High Computational Cost

DistilBERT requires:
- GPU acceleration
- higher memory usage
- longer training time

compared to classical ML models.

---

### 2. More Complex Training Pipeline

Transformer workflows involve:
- tokenization
- attention masks
- pretrained checkpoints
- fine-tuning procedures

which increases implementation complexity.

---

### 3. Lower Explainability

Transformer models are often:
- less interpretable

compared to linear ML models.

Understanding:
- why predictions occur
- which features dominate

is more difficult in transformer architectures.

---

# Key Observations from the Study

## 1. Classical NLP Performed Exceptionally Well

The project demonstrated that:
- TF-IDF + Linear Models

remain highly effective for structured text classification tasks.

Especially:
- Linear SVM
- Logistic Regression
- Naive Bayes

achieved near-transformer-level performance.

---

## 2. DistilBERT Matched the Best Classical Model

DistilBERT achieved:
- contextual understanding
- semantic learning
- transfer learning capability

while matching the accuracy of the best-performing classical model.

This highlights the power of:
- transformer-based NLP systems

---

## 3. Linear SVM Remained Highly Competitive

Despite the rise of transformers:
- Linear SVM

remained extremely competitive due to:
- sparse TF-IDF optimization
- high-dimensional linear separability
- efficient handling of text classification tasks

---

## 4. Random Forest Was Less Effective

Tree-based models performed comparatively weaker because:
- sparse TF-IDF matrices are not ideal for decision trees
- high-dimensional text spaces reduce tree efficiency

This confirms that:
- linear classifiers are generally better suited for classical NLP tasks.

---

# Business Perspective

Both classical NLP and transformer-based NLP models can significantly improve FlipItNews through:
- automated article categorization
- personalized recommendations
- scalable content management
- intelligent news organization
- improved search relevance

---

## Classical NLP Business Advantages

Classical ML models offer:
- lower infrastructure cost
- faster deployment
- lower computational requirements
- simpler maintenance

making them suitable for:
- lightweight production systems
- resource-constrained environments

---

## Transformer NLP Business Advantages

Transformer models offer:
- superior contextual understanding
- improved semantic comprehension
- better adaptability to complex language patterns

making them highly suitable for:
- advanced recommendation systems
- semantic search
- intelligent personalization
- large-scale NLP platforms

---

# Final Conclusion

This project successfully demonstrated both:
- Classical NLP
- Advanced Transformer-Based NLP

approaches for automated news classification.

Key findings include:
- TF-IDF combined with linear classifiers remains highly effective
- transformer models provide strong contextual understanding
- transfer learning significantly enhances NLP capability
- preprocessing and feature engineering play a critical role in model success

Among all evaluated models:
- Tuned Linear SVM
- DistilBERT

achieved the best overall classification performance.

The study confirms that:
- both classical and transformer-based NLP approaches

can effectively solve large-scale news classification problems, with the final choice depending on:
- computational resources
- scalability requirements
- contextual understanding needs
- business objectives

---

In [ ]:
# =========================
# Model Accuracy Comparison
# =========================

# Model names
models = [
    'Naive Bayes',
    'Logistic Regression',
    'Linear SVM',
    'Random Forest',
    'DistilBERT'
]

# Corresponding accuracies
accuracies = [
    97.65,
    97.42,
    97.89,
    95.77,
    97.89
]

# Plot
plt.figure(figsize=(10,6))

bars = plt.bar(models, accuracies)

# Add accuracy labels
for bar in bars:

    height = bar.get_height()

    plt.text(
        bar.get_x() + bar.get_width()/2,
        height + 0.05,
        f'{height:.2f}%',
        ha='center',
        fontsize=11
    )

# Labels and title
plt.ylabel('Accuracy (%)')

plt.xlabel('Models')

plt.title('Comparison of Model Accuracies')

plt.ylim(94, 99)

plt.show()

# Model Accuracy Comparison

The accuracy comparison demonstrates that all models achieved strong classification performance on the FlipItNews dataset.

Among all evaluated models:
- Tuned Linear SVM
- DistilBERT

achieved the highest accuracy of:
- 97.89%

This indicates that both:
- classical linear NLP models
- advanced transformer-based NLP models

are highly effective for automated news classification tasks.

Random Forest achieved comparatively lower accuracy, suggesting that linear classifiers and transformer models are better suited for sparse high-dimensional textual data.

Overall, the results confirm the effectiveness of:
- TF-IDF + Linear Classifiers
- Transformer-based contextual embeddings

for NLP-based text classification.

---

# Real-World Prediction Demo
Using Tuned Linear SVM due to its simplicity


In [ ]:
# =========================
# Sample News Articles
# =========================

sample_news = [

    "The government announced new election reforms and economic policies today.",

    "The football team secured a dramatic victory in the championship final.",

    "Apple launched its latest AI-powered smartphone and advanced chipset.",

    "The movie won several international awards at the film festival.",

    "Global stock markets experienced major growth after the company earnings report."
]

In [ ]:
# Transform text using TF-IDF

sample_tfidf = tfidf_vectorizer.transform(sample_news)

In [ ]:
# Predicted categories

predicted_categories = predictions

predicted_categories

In [ ]:
# Display Predictions

for news, category in zip(sample_news, predicted_categories):

    print("News Article:")
    print(news)

    print("Predicted Category:", category)

    print("-" * 80)

# Real-World News Classification Demo

To evaluate the practical applicability of the developed NLP pipeline, several unseen real-world news headlines were tested using the best-performing model.

The tuned Linear SVM classifier successfully predicted categories for news articles related to:
- Politics
- Sports
- Technology
- Entertainment
- Business

---

# Sample Predictions

| News Article | Predicted Category |
|---|---|
| The government announced new election reforms and economic policies today. | Business |
| The football team secured a dramatic victory in the championship final. | Sports |
| Apple launched its latest AI-powered smartphone and advanced chipset. | Technology |
| The movie won several international awards at the film festival. | Entertainment |
| Global stock markets experienced major growth after the company earnings report. | Business |

---

# Interpretation

The model successfully generalized to unseen news headlines and correctly identified the contextual theme of most articles.

A slight overlap was observed between:
- Politics
- Business

because political and economic news often share similar vocabulary and contextual patterns.

---

# Business Relevance

This demonstrates that the developed NLP classification system can effectively support:
- automated news categorization
- personalized content recommendations
- scalable content organization
- intelligent news management systems

The successful prediction of unseen articles confirms the practical deployment capability of the developed solution.

---

# Recommendations

Based on the findings from the NLP analysis and model evaluation, several actionable recommendations can help FlipItNews improve automated news categorization, content relevance, and overall user experience.

---

# 1. Deploy Linear SVM or DistilBERT for Automated News Classification

The study demonstrated that:
- Tuned Linear SVM
- DistilBERT

achieved the highest classification accuracy.

Therefore, FlipItNews can deploy these models to:
- automatically categorize incoming news articles
- reduce manual tagging effort
- improve operational efficiency
- ensure consistent content organization

Linear SVM can be preferred for:
- lightweight deployment
- lower computational cost

while DistilBERT can be used for:
- advanced contextual understanding
- semantic-aware classification systems

---

# 2. Improve Personalized News Recommendations

Accurate article categorization enables FlipItNews to:
- recommend category-specific content
- personalize user news feeds
- improve reader engagement
- increase user retention

For example:
- sports readers can receive sports-focused recommendations
- business readers can receive financial and market-related updates

This improves content relevance and user satisfaction.

---

# 3. Enhance Content Discovery and Search

Automated NLP categorization can improve:
- search relevance
- topic-based filtering
- intelligent content retrieval

Users can quickly discover relevant articles based on:
- category
- topic
- contextual meaning

This can significantly enhance the platform’s usability.

---

# 4. Expand and Diversify News Categories

The study showed slight overlap between categories such as:
- Business
- Politics
- Technology

FlipItNews can improve classification granularity by:
- introducing subcategories
- creating topic-specific labels
- diversifying news taxonomy

Examples include:
- AI Technology
- Financial Politics
- International Sports
- Entertainment Reviews

This can improve classification precision and recommendation quality.

---

# 5. Utilize Transformer Models for Advanced NLP Applications

DistilBERT demonstrated strong contextual understanding and semantic learning capability.

Transformer-based NLP models can further support:
- semantic search
- contextual recommendation systems
- trend analysis
- intelligent summarization
- sentiment-aware content analysis

This can help FlipItNews build more advanced AI-driven news platforms.

---



# Feedback Loop for Continuous Improvement

News content evolves continuously, and NLP models must adapt to:
- emerging vocabulary
- trending topics
- changing writing styles
- evolving user interests

Therefore, implementing a continuous feedback loop is essential for maintaining long-term model effectiveness.

---

# 1. Regular Model Retraining

The classification models should be periodically retrained using:
- newly published news articles
- updated category labels
- recent linguistic patterns

Regular retraining helps:
- maintain classification accuracy
- reduce model drift
- adapt to changing news trends

---

# 2. Incorporate User Feedback

User interaction data can be leveraged to improve model performance.

Examples include:
- article clicks
- reading preferences
- manual category corrections
- user engagement metrics

This feedback can help:
- refine classification models
- improve recommendation systems
- better understand user interests

---

# 3. Monitor Misclassified Articles

Analyzing incorrectly classified articles can help identify:
- overlapping categories
- ambiguous vocabulary
- emerging topics
- weaknesses in the NLP pipeline

This allows continuous improvement of:
- preprocessing strategies
- labeling quality
- model architecture

---

# 4. Update Vocabulary and Training Data

News language evolves rapidly due to:
- technological advancements
- political events
- social media trends
- global developments

The training dataset should therefore be regularly updated to include:
- new terminology
- emerging entities
- trending topics
- evolving contextual patterns

---

# 5. Hybrid NLP Deployment Strategy

A hybrid deployment approach can be adopted:
- Linear SVM for fast and lightweight classification
- DistilBERT for deeper contextual analysis

This balances:
- computational efficiency
- scalability
- contextual understanding
- production performance

---

# Overall Insight

The study confirms that combining:
- NLP preprocessing
- TF-IDF feature engineering
- machine learning classification
- transformer-based contextual embeddings

can significantly improve automated news categorization systems.

With continuous retraining and feedback integration, FlipItNews can build a scalable and adaptive AI-powered news recommendation ecosystem.

---

# Final Conclusion

This project successfully developed and evaluated an end-to-end NLP-based news classification system for the FlipItNews platform.

The study explored both:
- classical machine learning NLP techniques
- advanced transformer-based NLP models

to automate multi-class news categorization across:
- Business
- Entertainment
- Politics
- Sports
- Technology

---

# Major Achievements of the Project

The project successfully implemented:
- text preprocessing
- tokenization
- stopword removal
- lemmatization
- TF-IDF vectorization
- Bag of Words representation
- classical machine learning models
- hyperparameter tuning
- transformer-based NLP models

A comprehensive NLP pipeline was built and evaluated using multiple performance metrics and visualization techniques.

---

# Key Findings

Among the classical machine learning models:
- Tuned Linear SVM achieved the highest performance

Among advanced NLP approaches:
- DistilBERT achieved performance equal to the best classical model

Both models achieved:
- Accuracy = 97.89%

The results demonstrate that:
- TF-IDF combined with linear classifiers remains highly effective for structured text classification
- transformer-based models provide strong contextual understanding and semantic learning capability

---

# Business Impact

The developed NLP system can help FlipItNews:
- automate news categorization
- improve personalized recommendations
- enhance content discovery
- reduce manual tagging effort
- improve user engagement
- support scalable content management

The implementation of advanced NLP techniques can significantly improve operational efficiency and user experience.

---

# Overall Insight

This project demonstrates that combining:
- NLP preprocessing
- feature engineering
- machine learning
- transformer-based contextual embeddings

can effectively solve large-scale text classification problems.

The study also highlights the importance of:
- proper preprocessing
- model tuning
- contextual language understanding
- continuous model improvement

in building robust NLP systems.

---

# Future Scope

Although the developed NLP pipeline achieved excellent performance, several opportunities exist for further enhancement and future research.

---

# 1. Fine-Tuning Larger Transformer Models

Future work can explore more advanced transformer architectures such as:
- BERT
- RoBERTa
- XLNet
- DeBERTa

These models may further improve contextual understanding and classification performance.

---

# 2. Multilingual News Classification

The current system primarily focuses on English-language news articles.

Future enhancements can include:
- multilingual NLP support
- cross-lingual transformers
- regional language classification

to support a broader global audience.

---

# 3. Real-Time News Classification

The developed pipeline can be extended into:
- real-time streaming systems
- live news categorization engines
- scalable production deployment pipelines

This would enable instant categorization of incoming news articles.

---

# 4. Sentiment Analysis Integration

Future systems can combine:
- news categorization
- sentiment analysis

to better understand:
- public opinion
- emotional tone
- audience reactions

This can enhance recommendation quality and user engagement.

---

# 5. Personalized Recommendation Systems

The NLP models can be integrated with:
- recommendation engines
- user preference modeling
- behavioral analytics

to create highly personalized news feeds for users.

---

# 6. Continuous Learning and Feedback Systems

Future improvements may include:
- active learning
- continuous retraining
- user feedback integration
- adaptive learning systems

to ensure the model remains updated with:
- evolving news trends
- emerging vocabulary
- changing user interests

---

# Final Overall Insight

The project successfully demonstrated both:
- Classical NLP
- Advanced Transformer-Based NLP

approaches for automated news classification.

The findings confirm that modern NLP techniques can significantly improve:
- content organization
- recommendation quality
- semantic understanding
- scalability of news platforms

The developed solution provides a strong foundation for building intelligent AI-powered news recommendation and content management systems.

---